# [1.5.2] Grokking과 Modular Arithmetic (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/32_[1.5.2]_Grokking_&_Modular_Arithmetic)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part52_grokking_and_modular_arithmetic/1.5.2_Grokking_&_Modular_Arithmetic_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part52_grokking_and_modular_arithmetic/1.5.2_Grokking_&_Modular_Arithmetic_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 이번 장의 학습 자료에 관한 질문은 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 가는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

> *참고 - 어느 시점에서든 numpy 관련 에러(아마도 `module 'numpy.linalg._umath_linalg' has no attribute '_ilp64'`)가 발생하면, 커널을 재시작하고 설정 코드를 다시 실행하십시오. 그러면 에러가 해결될 것입니다.*

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-15-2.png" width="350">

# 소개

오늘의 목표는 모듈러 덧셈(modular addition)을 학습한 단일 레이어 transformer를 역공학(reverse-engineer)하는 것입니다! 이 작업을 담당하는 circuit은 이산 푸리에 변환(discrete Fourier transforms)과 삼각함수 항등식(trigonometric identities)을 포함하고 있다는 것이 밝혀졌습니다. 이는 지금까지 완전히 역공학된 알고리즘 작업 해결 circuit 중 아마도 가장 흥미로운 사례일 것입니다.

이 연습 문제들은 Neel Nanda와 Tom Lierberum의 [original notebook](https://colab.research.google.com/drive/1F6_1_cWXE5M7WocUcpQWp3v8z4b1jL20) (그리고 어느 정도는 [accompanying paper](https://arxiv.org/abs/2301.05217))에서 발췌하여 수정되었습니다. 우리는 grokking 결과의 재현보다는, 주로 이 토이 모델의 mechanistic 분석에 집중할 것입니다 (재현 작업은 이후 연습 문제에서 다뤄질 수 있습니다).

## 문제 설정

오늘 우리가 역공학(reverse-engineering)할 모델은 layer norm이 없고 학습 가능한 positional embedding을 가진 단일 레이어 transformer입니다. $d_{model} = 128$, $n_{heads} = 4$, $d_{head}=32$, $d_{mlp}=512$.

이 모델이 학습한 작업은 소수 $p = 113$에 대한 modulo 덧셈입니다. 입력 형식은 세 개의 token 시퀀스 `[x, y, =]`이며, $d_{vocab}=114$ ($0$에서 $p - 1$ 사이의 정수 및 $=$)로 구성됩니다. `=` 이후의 다음 token에 대한 예측은 $x + y \pmod{p}$에 해당하는 token이어야 합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/basic_schematic.png" width="480">

이 모델은 전체 데이터의 0.3을 학습 데이터로 사용하여 full batch training으로 학습되었습니다. AdamW를 사용하여 학습되었으며, $lr=10^{-3}$ 및 매우 높은 weight decay ($wd=1$)가 적용되었습니다.

## 알고리즘 요약

broadly, 이 알고리즘은 다음과 같이 작동합니다:

* 두 개의 token $x, y \in \{0, 1, \ldots, p - 1\}$이 주어지면, 이를 $\sin(\omega x)$, $\cos(\omega x)$, $\sin(\omega y)$, $\cos(\omega y)$로 매핑하며, 여기서 $\omega = \omega_k = \frac{2k\pi}{p}, k \in \mathbb{N}$입니다.
    * 다시 말해, 대부분의 주파수를 버리고 $k$의 특정 값에 해당하는 소수의 **key frequencies**만 유지합니다.
* 다음과 같은 quadratic terms를 계산합니다:
    $$
    \begin{align*}
    \cos(\omega x) &\cos(\omega y)\\
    \sin(\omega x) &\sin(\omega y)\\
    \cos(\omega x) &\sin(\omega y)\\
    \sin(\omega x) &\cos(\omega y)
    \end{align*}
    $$
    이는 (attention과 ReLU를 사용하는) 임시적인 방법으로 계산됩니다. 이를 통해 다음과 같은 linear combinations 또한 계산할 수 있습니다:
    $$
    \begin{align*}
    \cos(\omega (x+y)) &= \cos(\omega x) \cos(\omega y) - \sin(\omega x) \sin(\omega y)\\
    \sin(\omega (x+y)) &= \sin(\omega x) \cos(\omega y) + \cos(\omega x) \sin(\omega y)
    \end{align*}
    $$

* 출력 logit 벡터를 계산하며, 이때 각 요소 $\text{logits}[z]$는 다음과 같은 형태의 항들의 linear combination입니다:
    $$
    \cos(\omega (x + y - z)) = \cos(\omega (x+y)) \cos(\omega z) + \sin(\omega (x+y)) \sin(\omega z)
    $$
    이는 위에서 계산한 항들의 linear combination입니다.
* 이러한 값들(서로 다른 $k$에 대해)이 모두 더해져 최종 출력이 됩니다.
    * $z^* = x + y \; (\operatorname{mod} p)$ 위치에는 [constructive interference](https://en.wikipedia.org/wiki/Wave_interference)가 존재하고, 그 외의 모든 곳에서는 destructive interference가 발생합니다. 따라서 정확한 예측을 얻을 수 있습니다.

## 표기법

모호함을 없애기 위해, 이번 실습에서 사용할 표기법에 대해 몇 가지 설명하겠습니다:

* $x$ 및 $y$은 항상 모델의 두 입력을 나타냅니다. 또한 때때로 이 입력들의 one-hot encoding인 $t_0$ 및 $t_1$이라는 용어를 사용하겠습니다.
    * 세 번째 입력 token인 `=`는 항상 $t_2$로 지칭됩니다. $t_0$ 및 $t_1$과 달리, 이 token은 모든 입력 sequence에서 항상 동일합니다.
    * $t$은 세 개의 one-hot encoded token 전체를 담은 행렬을 나타내며, 즉 크기가 $(3, d_{vocab})$입니다. 여기서 우리는 $d_{vocab} = p + 1$을 가집니다 ($0$부터 $p - 1$까지의 모든 숫자와 token `=`이 있기 때문입니다).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/tensor.png" width="280">

* $z$는 항상 모델의 출력을 나타냅니다. 예를 들어, 모델이 "$\cos(\omega (x + y - z))$을 계산한다"고 말할 때, 이는 출력 logit 벡터가 다음과 같은 sequence임을 의미합니다:

    $$
    (\cos(\omega (x + y - z)))_{z = 0, 1, ..., p-1}
    $$
    이때 `=` 부호에 대한 logit은 제외한다는 점에 유의하십시오.


* 우리는 TransformerLens의 관례에 따라 행렬을 왼쪽에 곱하는 방식을 유지합니다. 예를 들어:
    * embedding 행렬 $W_E$의 shape은 $(d_{vocab}, d_{model})$입니다.
    * $t_0 ^T W_E \in \mathbb{R}^{d_{model}}$은 첫 번째 token의 embedding입니다.
    * $t W_E \in \mathbb{R}^{3 \times d_{model}}$는 세 token 모두의 embedding입니다.

## 내용 및 학습 목표

### 1️⃣ 주기성 및 Fourier basis

이 섹션에서는 toy model에 익숙해지는 시간을 갖습니다. 초기 조사를 수행하며 activation이 매우 주기적이라는 것을 확인하게 됩니다. 또한, 주기 함수를 표현하기 위해 Fourier basis를 사용하는 방법을 배웁니다.

> ##### 학습 목표
>
> * 문제 정의, 모델 architecture, 그리고 가능한 솔루션들의 대응하는 함수 형태를 이해합니다.
> * Fourier basis(1D 및 2D)와 이를 사용하여 임의의 함수를 표현하는 방법을 배웁니다.
> * 주기 함수가 Fourier basis에서 sparse하다는 점과 이것이 모델의 weight와 어떻게 연관되는지 이해합니다.

### 2️⃣ Circuits 및 Feature 분석

이 섹션에서는 Fourier basis에 대한 이해와 모델 weight의 주기성을 적용하여, 모델이 태스크를 해결하기 위해 사용하는 정확한 알고리즘을 분석합니다. 여러 가지 다양한 방법으로 가설을 검증하게 됩니다.

> ##### 학습 목표
>
> * 1D 및 2D Fourier basis에 대한 이해를 적용하여, 모델의 activation / effective weight가 Fourier basis에서 매우 sparse함을 보입니다.
> * 이러한 관찰 결과를 모델 알고리즘에 대한 구체적인 가설로 전환합니다.
> * 통계적 방법과 ablation과 같은 intervention을 사용하여 이러한 가설들을 검증합니다.
> * 모델의 알고리즘과 그것이 태스크를 해결하는 방식을 완전히 이해합니다.

### 3️⃣ 학습 중 분석

이 섹션에서는 학습 과정 동안 모델이 어떻게 진화하는지 살펴봅니다. 이 섹션은 선택 사항이며, 여기서 도출하는 관찰 결과들은 다른 자료들에 비해 더 추측에 가깝습니다.

> ##### 학습 목표
>
> * 시간에 따른 metric을 추적하는 개념과, 이를 통해 특정 circuit이 언제 형성되는지 어떻게 알 수 있는지 이해합니다.
> * 모델 weight matrix의 singular value가 시간에 따라 어떻게 진화하는지 조사하고 해석합니다.
> * 교환법칙(commutativity)과 같이 모델 내의 다른 능력들이 어떻게 형성되는지 조사합니다.

### ☆ 보너스

마지막으로, 이러한 연습 문제들에 대한 토론과 향후 나아갈 방향에 대한 생각으로 마무리합니다.

## 설정 코드

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

import einops
import numpy as np
import torch as t
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from jaxtyping import Float
from torch import Tensor
from tqdm import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.utils import to_numpy

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part52_grokking_and_modular_arithmetic"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

grokking_root = section_dir / "Grokking"
saved_runs_root = grokking_root / "saved_runs"

import part52_grokking_and_modular_arithmetic.tests as tests
import part52_grokking_and_modular_arithmetic.utils as utils

device = t.device("cuda" if t.cuda.is_available() else "cpu")

t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# 1️⃣ 주기성 & Fourier basis

> ##### 학습 목표
>
> * 문제 정의, 모델 architecture, 그리고 가능한 솔루션들의 대응하는 함수 형태를 이해합니다.
> * Fourier basis (1D 및 2D)에 대해 배우고, 이를 사용하여 임의의 함수를 어떻게 표현할 수 있는지 학습합니다.
> * 주기 함수가 Fourier basis에서 sparse하다는 점과, 이것이 모델의 weight와 어떻게 연관되는지 이해합니다.

## 모델 아키텍처

먼저, 모델과 몇 가지 유용한 activation 및 weight를 약어로 정의하겠습니다.

이전 페이지에서 제공된 정보를 복습해 보겠습니다:

> *오늘 우리가 역공학(reverse-engineering)할 모델은 layer norm이 없고 학습 가능한 positional embedding을 가진 단일 레이어 transformer입니다. $d_{model} = 128$, $n_{heads} = 4$, $d_{head}=32$, $d_{mlp}=512$.*
>
> *이 모델이 학습한 태스크는 소수 $p = 113$에 대한 modulo 덧셈입니다. 입력 형식은 세 개의 token 시퀀스 `[x, y, =]`이며, $d_{vocab}=114$ ($0$부터 $p - 1$까지의 정수 및 $=$)로 구성됩니다. `=` 이후의 다음 token에 대한 예측은 $x + y \pmod{p}$에 해당하는 token이어야 합니다.*

아래 코드를 실행하여 모델을 정의하십시오:

In [ ]:
p = 113

cfg = HookedTransformerConfig(
    n_layers=1,
    d_vocab=p + 1,
    d_model=128,
    d_mlp=4 * 128,
    n_heads=4,
    d_head=128 // 4,
    n_ctx=3,
    act_fn="relu",
    normalization_type=None,
    device=device,
)

model = HookedTransformer(cfg)

다음으로, 아래 코드를 실행하여 GitHub와 HuggingFace에서 데이터를 다운로드합니다.

In [ ]:
if not grokking_root.exists():
    os.system(f'git clone https://github.com/neelnanda-io/Grokking.git "{grokking_root.as_posix()}"')
    assert grokking_root.exists()
    os.mkdir(grokking_root / "large_files")

In [ ]:
REPO_ID = "callummcdougall/grokking_full_run_data"
FILENAME = "full_run_data.pth"

local_dir = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

full_run_data = t.load(local_dir, weights_only=True)
state_dict = full_run_data["state_dicts"][400]

model = utils.load_in_state_dict(model, state_dict)

이 과정이 완료되면, 다음의 helper 함수들을 사용하여 weight를 로드할 수 있습니다:

모델에 대해 mech interp를 시작하기 전에, loss 곡선을 살펴보겠습니다. 모델의 training 및 test loss 곡선이 얼마나 빠르게 내려가는지 확인해 봅시다.

In [ ]:
utils.lines(
    lines_list=[full_run_data["train_losses"][::10], full_run_data["test_losses"]],
    labels=["train loss", "test loss"],
    title="Grokking Training Curve",
    x=np.arange(5000) * 10,
    xaxis="Epoch",
    yaxis="Loss",
    log_y=True,
    width=900,
    height=450,
)

정말 흥미롭습니다! 모델이 처음에는 학습 데이터를 암기하지만(train loss 곡선은 거의 0까지 급격히 떨어지는 반면, test loss 곡선은 실제로 상승합니다), 결국 태스크를 "grok"하게 됩니다. 즉, 보지 못한 데이터에 대해 일반화하는 법을 갑자기 배우게 됩니다.

이번 섹션과 다음 섹션에서는 우리 모델을 사용하여 mech interp를 수행하는 데 집중하겠습니다. 세 번째 섹션에서는 이와 같은 그래프를 더 자세히 조사하고 시간이 지남에 따라 다른 metric들을 추적하겠습니다. 마지막 섹션에서는 이 작업의 더 높은 수준의 함의와 가능한 향후 방향에 대해 논의합니다.

## 헬퍼 변수

몇 가지 유용한 변수들을 정의하고, 예상한 대로 설정되었는지 확인하기 위해 그 shape을 출력해 보겠습니다:

In [ ]:
# Helper variables
W_O = model.W_O[0]
W_K = model.W_K[0]
W_Q = model.W_Q[0]
W_V = model.W_V[0]
W_in = model.W_in[0]
W_out = model.W_out[0]
W_pos = model.W_pos
W_E = model.W_E[:-1]
final_pos_resid_initial = model.W_E[-1] + W_pos[2]
W_U = model.W_U[:, :-1]

print("W_O  ", tuple(W_O.shape))
print("W_K  ", tuple(W_K.shape))
print("W_Q  ", tuple(W_Q.shape))
print("W_V  ", tuple(W_V.shape))
print("W_in ", tuple(W_in.shape))
print("W_out", tuple(W_out.shape))
print("W_pos", tuple(W_pos.shape))
print("W_E  ", tuple(W_E.shape))
print("W_U  ", tuple(W_U.shape))

여기서 주의할 점은, `=` token에 해당하는 마지막 행과 열을 제거하기 위해 embedding 및 unembedding matrix의 slice를 취했다는 것입니다. 이렇게 한 이유는 나중에 이 weight들에 대해 Fourier transform을 수행하기 위해서입니다. 이제부터 $W_E$ 및 $W_U$를 언급할 때는 보통 이 더 작은 matrix들을 의미합니다. `final_pos_resid_initial`은 나중에 (시퀀스 위치 2에 대한 query vector를 얻기 위해) 필요하므로 명시적으로 정의했습니다.

또한, 많은 matrix들을 `[0]`로 인덱싱했다는 점에 유의하십시오. 이는 첫 번째 차원이 layer 차원이며, 우리 모델은 단 하나의 layer만 가지고 있기 때문입니다.

다음으로, 모든 데이터에 대해 모델을 실행하겠습니다. 여기서 우리가 무엇을 하고 있는지 명확히 할 필요가 있습니다. 가능한 모든 $p^2 = 113^2 = 12769$개의 시퀀스를 각각 가져와 하나의 batch로 쌓은 다음, 모델을 실행하는 것입니다. 이는 이 특정 문제에서 우리가 다루는 범위가 매우 작기 때문에 가능합니다. 모든 중간 activation을 저장하기 위해 `run_with_cache` 방법을 사용하겠습니다.

In [ ]:
# Get all data and labels, and cache activations
all_data = t.tensor([(i, j, p) for i in range(p) for j in range(p)]).to(device)
labels = t.tensor([utils.target_fn(i, j) for i, j, _ in all_data]).to(device)
original_logits, cache = model.run_with_cache(all_data)

# Final position only, also remove the logits for `=`
original_logits = original_logits[:, -1, :-1]

# Get cross entropy loss
original_loss = utils.cross_entropy_high_precision(original_logits, labels)
print(f"Original loss: {original_loss.item():.3e}")

### 연습 문제 - 핵심 activation 추출하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-10 minutes on these exercises.
> These are just designed to re-familiarize yourself with the ActivationCache object and how to use it.
> ```

나중에 조사하게 될 몇 가지 중요한 activation으로는 attention 행렬과 neuron activation이 있습니다. 아래 코드에서 다음 항목들을 정의해야 합니다:

* `attn_mat`: 모든 sequence에 대해, query token `=`에 대한 attention pattern입니다. 이 텐서의 shape은 `(batch, head, key_posn)`이어야 하며, 이 경우에는 `(12769, 4, 3)`입니다.
    * 우리가 분류 결과를 얻는 위치가 바로 이곳이기 때문에, `=`가 query token인 경우에만 관심을 가집니다.
* `neuron_acts_post`: ReLU 함수를 적용한 후, **마지막 sequence position에 대한** neuron activation입니다. 이 텐서의 shape은 `(batch, d_mlp)`이어야 하며, 이 경우에는 `(12769, 512)`입니다.
    * 다시 한번, 우리는 마지막 sequence position에만 관심을 가집니다. 그 이유를 알 수 있습니까?
* `neuron_acts_pre`: 위와 동일하지만, ReLU를 적용하기 전의 값입니다.

텐서의 shape을 출력하여 결과를 확인할 수 있습니다.

In [ ]:
# YOUR CODE HERE - get the relevant activations

# Test shapes
assert attn_mat.shape == (p * p, cfg.n_heads, 3)
assert neuron_acts_post.shape == (p * p, cfg.d_mlp)
assert neuron_acts_pre.shape == (p * p, cfg.d_mlp)

# Test values
tests.test_cache_activations(attn_mat, neuron_acts_post, neuron_acts_pre, cache)

<details><summary>솔루션</summary>

```python
attn_mat = cache["pattern", 0][:, :, 2]
neuron_acts_post = cache["post", 0][:, -1]
neuron_acts_pre = cache["pre", 0][:, -1]
```
</details>

## 함수 형태

다음으로, 우리 모델 솔루션의 함수 형태에 대해 생각해보겠습니다.

### 연습 문제 - 초기 질문에 답하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20-30 minutes on these exercises.
> Thinking about the functional form of your model before you start analysing a toy problem is an important skill.
> ```

문제에 대해 생각하고 이것이 모델의 내부 구조와 어떻게 연관되는지 고민해 볼 수 있도록 몇 가지 질문을 준비했습니다. 아래 드롭다운에서 모든 질문에 대한 답변(및 다른 지점들에 대한 더 자세한 논의)을 확인하실 수 있습니다.

* 모델에 입력되는 6가지의 서로 다른 정보(3개의 token embedding과 3개의 positional embedding) 중에서, 모듈러 덧셈 작업을 해결하는 데 관련이 있는 것은 무엇입니까?
* 이것이 position embedding의 역할에 대해 시사하는 바는 무엇입니까?
* attention pattern은 어떤 모습이어야 합니까? attention pattern의 어느 부분이 실제로 중요할까요?
* direct path(즉, MLP나 attention 없이 embedding -> unembedding으로 이어지는 경로)의 역할은 무엇일까요? attention layer는 거치지 않고 MLP layer만 거치는 경로는 어떠할까요?
* 모델에서 어떤 종류의 대칭성(symmetries)이 나타날 것으로 예상하십니까?

<details>
<summary>답변</summary>

덧셈은 교환 법칙이 성립하므로 position embedding은 무관합니다. 실제로, 이로 인해 이러한 position embedding들은 거의 대칭적인 모습을 보입니다. 마지막 token은 항상 `=`이므로, 처음 두 token의 token embedding만이 관련이 있습니다. position `2`은 상수 embedding을 가지며 관련 정보를 제공하지 않으므로, attention pattern은 position `2`이 position `0`과 `1`에만 attention을 기울이는 형태여야 합니다. (다만, 이것이 bias 항으로 작용할 수는 있습니다. 하지만 훈련 중 강한 weight decay를 사용하기 때문에 이는 권장되지 않으며, 경험적으로도 발생하지 않습니다.)

direct path는 관련 정보를 제공하지 않으며 따라서 bias 항으로만 작용합니다. 경험적으로, unembedding matrix를 적용하기 전에 residual stream을 0으로 ablation 해도 성능에 큰 영향을 주지 않습니다. attention layer를 거치지 않고 MLP layer만 거치는 경로도 마찬가지입니다 (정보가 `x`, `y` token에서 예측에 사용할 token으로 이동할 수 없기 때문입니다).

앞서 언급했듯이 덧셈은 교환 법칙이 성립하므로, 모델이 처음 두 token을 처리하는 방식 사이에 대칭성이 있을 것으로 예상합니다. 이에 대한 증거는 다음과 같습니다:

* pos 0과 pos 1의 position embedding 간의 차이를 살펴보면, 두 값이 서로 가깝고 cosine similarity가 높다는 것을 알 수 있습니다.
* neuron activation과 neuron activation의 전치 행렬(transpose) 간의 차이를 살펴보면 (즉, $N(x, y)$과 $N(y, x)$를 비교), 두 값이 서로 가깝다는 것을 알 수 있습니다.

```python
# Get the first three positional embedding vectors
W_pos_x, W_pos_y, W_pos_equals = W_pos

# Look at the difference between positional embeddings; show they are symmetric
def compare_tensors(v, w):
    return ((v-w).pow(2).sum()/v.pow(2).sum().sqrt()/w.pow(2).sum().sqrt()).item()
print('Difference in position embeddings', compare_tensors(W_pos_x, W_pos_y))
print('Cosine similarity of position embeddings', t.cosine_similarity(W_pos_x, W_pos_y, dim=0).item())

# Compare N(x, y) and N(y, x)
neuron_acts_square = neuron_acts.reshape(p, p, d_mlp)
print('Difference in neuron activations for (x,y) and (y,x): {.2f}'.format(
    compare_tensors(
        neuron_acts_square,
        einops.rearrange(neuron_acts_square, "x y d_mlp -> y x d_mlp")
    )
))
```

덧셈은 교환 법칙이 성립하므로 이는 타당합니다! Position 0과 1은 대칭적이어야 합니다.

position 2에서 자기 자신으로 향하는 attention이 무시할 수 있을 정도로 작다는 증거입니다 - 모든 데이터 포인트에 대해 각 head의 각 position에 대한 평균 attention을 플롯해 보면, $2\to 2$의 평균이 거의 0에 가깝고 (attention은 항상 양수이므로 거의 항상 0에 가깝습니다), 대칭성에서 예상했듯이 $2\to 0$와 $2 \to 1$ 모두 평균이 0이 되는 것을 볼 수 있습니다.

```python
imshow(attn_mat.mean(0), xaxis='Position', yaxis='Head', title='Average Attention by source position and head', text_auto=".3f")
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/half_attn_2.png" width="800">

*(이러한 attention pattern을 플롯하기 위해 circuitsvis를 사용할 수도 있지만, 여기서는 attention pattern 분석이 그리 복잡하지 않으므로 Plotly를 사용해도 무방합니다.)*
</details>

### 연습 문제 - 함수 형태 유도하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 20-30 minutes on these exercises.
> This exercise is challenging, and involves only maths and no coding. You should look at parts of the solution if you get stuck, because there are several steps involved. Even if you can't get all the steps, any progress is good!
> ```

이전 연습 문제에서 접하셨겠지만, [comprehensive mathematical framework for understanding transformer circuits](https://transformer-circuits.pub/2021/framework/index.html)이 존재합니다. 하지만 오늘은 모듈러 덧셈을 담당하는 circuit을 이해하기 위해 그 프레임워크의 모든 기능을 사용할 필요는 없습니다. 우리 모델이 단일 레이어로만 구성되어 있고, 우리가 해결하려는 작업에 활용할 수 있는 특별한 구조가 있기 때문입니다.

우리 모델에 대해 다음과 같은 단순화된 가정을 고려해 보겠습니다:

* position embedding은 무관하며, 성능 저하 없이 zero-ablation 할 수 있습니다.
* residual stream은 무관하며, 성능 저하 없이 mean-ablation 할 수 있습니다.
* 모든 head에 대해, position `2`는 position `0`과 `1`에만 attention을 줍니다.

모델에 의해 계산되는 함수 $\ell = f(t)$를 작성하십시오. 여기서 $\ell \in \mathbb{R}^p$은 각 token에 대한 logit 벡터이고, $t \in \mathbb{R}^{2 \times p}$은 입력 정수 $m$와 $n$를 나타내는 one-hot 벡터입니다. 얻어진 식을 가능한 한 단순화하십시오. 이에 대해 무엇을 말할 수 있습니까?

<details>
<summary>힌트 - 다이어그램</summary>

다음은 서로 다른 계산 단계들을 더 명시적으로 보여주는 다이어그램입니다. 이를 사용하여 $f(t)$에 대한 closed-form 식을 작성할 수 있습니까?

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/functional_form.png" width="800">
</details>

<details>
<summary>힌트 - 첫 단계들</summary>

솔루션은 다음과 같은 형태가 될 것입니다:

$$
f(t) = \operatorname{MLP}(\operatorname{Attn}(tW_E)_2)W_U
$$

여기서 $t \in R^{3 \times p}$은 1-hot 인코딩된 token 벡터들이고, $\operatorname{MLP}$은 MLP 레이어(각 시퀀스 위치의 residual stream 벡터에 동일하게 작용함)를 나타내며, $\operatorname{Attn}$는 attention 레이어를 나타냅니다(예측값을 여기서 가져오므로 시퀀스 위치 2에서의 값을 취합니다).

여기서 실제 행렬들을 사용하여 $\operatorname{MLP}$와 $\operatorname{Attn}(\cdot)_2$를 작성할 수 있습니까? token 2가 token 0과 1에만 attention을 준다고 가정하여 단순화할 수 있습니까?
</details>

<details>
<summary>정답</summary>

모델을 단계별로 살펴보겠습니다. $n_\mathrm{seq} = 3$를 시퀀스 길이로 정의하면, (one-hot 인코딩된) 입력 token은 $t \in \mathbb{R}^{n_\mathrm{seq} \times p}$이 됩니다. 여기에는 one-hot 인코딩된 정수 $t_0$와 $t_1$, 그리고 (모든 입력에 대해 동일한) one-hot 인코딩된 등호 $t_2$가 포함됩니다. embedding 행렬 $W_E \in \mathbb{R}^{p \times d_\mathrm{model}}$를 적용하면 다음과 같은 embedding을 얻습니다:

$$
v = t W_E \in \mathbb{R}^{n_\mathrm{seq} \times d_\mathrm{model}}.
$$

우리의 함수는 다음과 같은 형태가 될 것입니다:

$$
f(t) = \operatorname{MLP}(\operatorname{Attn}(v)_2)W_U
$$

여기서 $\operatorname{MLP}$은 MLP 레이어(각 시퀀스 위치의 residual stream 벡터에 동일하게 작용함)를 나타내고, $\operatorname{Attn}$은 attention 레이어를 나타냅니다(예측값을 여기서 가져오므로 시퀀스 위치 2에서의 값을 취합니다). 참고로, 이는 다른 모든 residual stream 항들을 무시한 것입니다. 중요할 수 있는 다른 경로는 attention 레이어는 거치지만 MLP는 거치지 않는 경로들이지만, 두 레이어를 모두 거치는 경로가 훨씬 더 중요할 것이라고 추측할 수 있습니다.

먼저 더 간단한 MLP부터 다루겠습니다. 함수 형태는 다음과 같습니다:

$$
\operatorname{MLP}(w) = \operatorname{ReLU}\left(w^T W_{in}\right)W_{out}
$$

여기서 $w \in \mathbb{R}^{d_\mathrm{model}}$은 $\operatorname{Attn}$을 적용한 후의 residual stream 내 벡터(즉, 특정 시퀀스 위치의 벡터)입니다.

이제 attention에 대해 생각해보겠습니다. 다음과 같이 나타낼 수 있습니다:

$$
\begin{aligned}
\operatorname{Attn}(v)_2&=\sum_h \operatorname{softmax}\left(\frac{v_2^{\top} W_Q^h (W_K^h)^T\left[v_0 \, v_1\right]}{\sqrt{d_{head}}}\right)\left[v_0\, v_1\right]^T W_V^h W_O^h \\
&= \sum_h (\alpha^h v_0 + (1 - \alpha^h) v_1)^T W_V^h W_O^h \\
&\in \mathbb{R}^{d_{model}}
\end{aligned}
$$

여기서 $v_0, v_1, v_2 \in \mathbb{R}^{d_{model}}$은 residual stream에 있는 세 개의 embedding 벡터이며, $\alpha^h$는 head $h$에서 token 2가 token 0에 주는 attention 확률입니다. token 2가 자기 자신에게 주는 attention은 (거의 0인 것을 확인했으므로) 무시했습니다. 이것이 key 쪽 항인 $v = t W_E$을 처음 두 벡터 $\left[v_0 \, v_1\right]$로 대체한 이유이며, 따라서 softmax는 key 위치 $\{0, 1\}$에 대해서만 수행됩니다.

$\alpha^h$에 대한 공식을 단순화할 수 있을까요? 결과적으로 가능합니다. 2차원에 대해 softmax를 취하는 것은 logit 간의 차이에 sigmoid를 취하는 것과 동일합니다:

$$
\operatorname{softmax}\left(\begin{array}{c}
\alpha \\
\beta
\end{array}\right)=\left(\begin{array}{c}
e^\alpha / (e^\alpha+e^\beta) \\
e^\beta / (e^\alpha+e^\beta)
\end{array}\right) = \left(\begin{array}{c}
\sigma(\alpha-\beta) \\
1-\sigma(\alpha-\beta)
\end{array}\right)
$$

따라서 다음과 같이 작성할 수 있습니다:

$$
\begin{aligned}
\alpha^h &= \sigma\left(\frac{v_2^{\top} W_Q^h (W_K^h)^Tv_0}{\sqrt{d_{head}}} - \frac{v_2^{\top} W_Q^h (W_K^h)^Tv_1}{\sqrt{d_{head}}}\right) \\
&= \sigma\left(\frac{v_2^{\top} W_Q^h (W_K^h)^T(v_0 - v_1)}{\sqrt{d_{head}}}\right) \\
&= \sigma\left(\frac{(t_2^T W_E) W_Q^h (W_K^h)^T W_E^T(t_0 - t_1)}{\sqrt{d_{head}}}\right) \\
\end{aligned}
$$

이는 오직 가중치 행렬들과 one-hot 인코딩된 token $t_i$로만 표현됩니다.

이제 이 두 가지를 합쳐보겠습니다. 함수 형태는 다음과 같습니다:

$$
f(t)=\operatorname{ReLU}\left(\sum_n\left(\alpha^h t_x+\left(1-\alpha^h\right) t_y\right)^T W_E W_V^h W_O^h W_{in}\right) W_{out} W_U
$$

---

함수 형태를 구했으므로, 모델의 동작이 **effective weight matrices**라고 부르는 몇 개의 행렬에 의해 완전히 결정된다는 것을 알 수 있습니다. 그것들은 다음과 같습니다:

* $W_{logit} := W_{out} W_U$: 크기는 $(d_{mlp}, d_{vocab}-1) = $ `(512, p)`이며, 비선형 activation 함수의 출력에서 최종 logit까지 어떻게 도달하는지를 알려줍니다.

* $W_{neur} := W_E W_V W_O W_{in}$: 크기는 $(n_{heads}, d_{vocab}-1, d_{mlp}) =$ `(4, p, 512)`입니다 (각 head의 OV 행렬들을 0번째 차원을 따라 쌓은 것입니다). 이는 초기 embedding들의 가중 합에서 뉴런 activation까지 어떻게 도달하는지를 알려줍니다.

* $W_{attn} := (t_2^T W_E) W_Q^h (W_K^h)^T W_E^T / \sqrt{d_{head}}$: 크기는 $(n_{heads}, d_{vocab}-1) =$ `(4, p)`입니다. 이는 attention score를 얻기 위해 $(t_0 - t_1)$과 내적하는 행(head당 하나의 벡터)들의 집합입니다.

이들이 transformer 내에서 어떻게 작용하는지 확인할 수 있습니다:

$$
f(t)=\operatorname{ReLU}\Bigg(\sum_h\underbrace{\bigg(\alpha^h t_0\;+\;\left(1\;-\;\alpha^h\right) t_1 \bigg)^T}_{\textstyle{\alpha^h = \sigma(W_{attn}^h(t_0 - t_1))}}  \underbrace{W_E W_V^h W_O^h W_{in}}_{\textstyle{W_{neur}^h}}\Bigg) \;\underbrace{W_{out} W_U}_{\textstyle{W_{logit}}}
$$

참고 - 위의 $W_E$와 $W_U$은 대부분 축소된 행렬(따라서 크기가 $d_{vocab}-1$임)을 의미합니다. 이는 $t_0$와 $t_1$가 항상 정수 $0, 1, ..., p-1$만 될 수 있고, 우리가 관심을 갖는 유일한 logit 출력은 정수에 해당하는 것들이기 때문입니다. 유일한 예외는 $W_{attn}$을 정의할 때인데, $t_2^T W_E$ 항이 **전체 embedding 행렬의 마지막 행**과 같기 때문입니다.
</details>

### 연습 문제 - 유효 가중치 행렬 정의하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on these exercises.
> 
> They should not be challenging, and are designed to get you more comfortable with constructing circuits from model weights in a hands-on way.
> ```

위의 **Answers** 드롭다운에서, 우리는 transformer의 동작을 종합적으로 결정하는 세 가지 **유효 가중치 행렬(effective weight matrices)**을 확인했습니다. 아래에서 모델로부터 이 세 가지 행렬을 직접 계산해야 합니다. 분해된 행렬(factored matrices) 사용에 대해서는 걱정하지 마십시오. 모델의 크기가 크지 않으므로 그럴 필요가 없습니다 (이후의 모든 연습 문제도 마찬가지입니다).

In [ ]:
# YOUR CODE HERE - define the following matrices

# Test shapes
assert W_logit.shape == (cfg.d_mlp, cfg.d_vocab - 1)
assert W_neur.shape == (cfg.n_heads, cfg.d_vocab - 1, cfg.d_mlp)
assert W_attn.shape == (cfg.n_heads, cfg.d_vocab - 1)

# Test values
tests.test_effective_weights(W_logit, W_neur, W_attn, model)

<details>
<summary>풀이 설명</summary>

참고로, 이 모든 예제는 `@` 연산자를 사용합니다. `einsum`가 더 명시적이고 오류를 범할 가능성이 적어 더 편하게 느끼실 수 있지만, 다루고 있는 행렬의 크기와 `@`이 다양한 경우(예: 2D 초과 tensor가 포함된 경우)에 matrix multiplication을 어떻게 처리하는지 이미 잘 알고 계신다면 `@`를 사용하는 것도 전혀 문제가 없습니다.

예를 들어, `W_OV`은 첫 번째 차원이 head index인 3D tensor이며, 이를 `@`을 사용하여 2D matrix와 곱할 때 PyTorch는 `W_OV`를 matrix의 batch로 유용하게 해석하며, 이는 정확히 우리가 원하는 결과입니다.

하지만 몇 가지 미묘한 점이 있습니다. 예를 들어, 3D tensor에 `.T`을 사용하면 기본적으로 기대하시는 것처럼 마지막 두 차원을 transpose하지 않는다는 점을 기억하십시오. 이것이 우리가 대신 transpose 메서드를 필요로 하는 이유입니다.

```python
W_logit = W_out @ W_U
W_OV = W_V @ W_O
W_neur = W_E @ W_OV @ W_in
W_QK = W_Q @ W_K.transpose(-1, -2)
W_attn = final_pos_resid_initial @ W_QK @ W_E.T / (cfg.d_head ** 0.5)

```
</details>


<details><summary>정답</summary>

```python
W_logit = W_out @ W_U

W_OV = W_V @ W_O
W_neur = W_E @ W_OV @ W_in

W_QK = W_Q @ W_K.transpose(-1, -2)
W_attn = final_pos_resid_initial @ W_QK @ W_E.T / (cfg.d_head**0.5)
```
</details>

### 모든 것은 주기적입니다

activation과 위에서 언급한 effective weight matrix에 대한 초기 조사 및 시각화를 수행하면, vocab basis에서의 양상들이 분명히 주기적이라는 것을 알 수 있습니다. 아래 셀들을 실행하여 이를 직접 확인해 보시기 바랍니다.

#### Activations

**Attention patterns:**

아래 코드에서 생성된 heatmap은 $p\times p$ 이미지이며, 여기서 셀 $(x, y)$ 은 입력 $x$ 및 $y$ 에 대한 특정 activation(즉, 네트워크의 hidden layer에 있는 실수 값)을 나타냅니다.

**참고:** 애니메이션 슬라이더는 시간 차원이 아니라 서로 다른 head들을 나타내기 위해 사용됩니다.

**참고:** $A_{2\to 2}^h\approx 0$, 따라서 $A_{2\to 1}^h = 1-A_{2\to 0}^h$ 입니다. 이러한 이유로, 아래에서 가장 먼저 수행할 작업은 `attn_mat` 이 처음 두 token에 가해진 attention만을 참조하도록 재정의하는 것입니다.

**참고:** 먼저 attention matrix를 재배치하여, 처음 두 차원이 모듈로 연산 방정식의 (x, y) 좌표를 나타내도록 합니다. 이것이 플롯 축의 의미입니다.

In [ ]:
attn_mat = attn_mat[:, :, :2]  # we only care about attn from first 2 tokens to the "=" token

# We rearrange attn_mat, so the first two dims represent (x, y) in modular arithmetic equation
attn_mat_sq = einops.rearrange(attn_mat, "(x y) head seq -> x y head seq", x=p)

utils.inputs_heatmap(
    attn_mat_sq[..., 0],
    title="Attention score for heads at position 0",
    animation_frame=2,
    animation_name="head",
)

**Neuron activation:**

In [ ]:
# We rearrange activations, so the first two dims represent (x, y) in modular arithmetic equation
neuron_acts_post_sq = einops.rearrange(neuron_acts_post, "(x y) d_mlp -> x y d_mlp", x=p)
neuron_acts_pre_sq = einops.rearrange(neuron_acts_pre, "(x y) d_mlp -> x y d_mlp", x=p)

top_k = 3
utils.inputs_heatmap(
    neuron_acts_post_sq[..., :top_k],
    title=f"Activations for first {top_k} neurons",
    animation_frame=2,
    animation_name="Neuron",
)

**Effective weights:**

#### **$W_{neur}$**

In [ ]:
top_k = 5
utils.animate_multi_lines(
    W_neur[..., :top_k],
    y_index=[f"head {hi}" for hi in range(4)],
    labels={"x": "Input token", "value": "Contribution to neuron"},
    snapshot="Neuron",
    title=f"Contribution to first {top_k} neurons via OV-circuit of heads (not weighted by attention)",
)

#### **$W_{attn}$**

In [ ]:
utils.lines(
    W_attn,
    labels=[f"head {hi}" for hi in range(4)],
    xaxis="Input token",
    yaxis="Contribution to attn score",
    title="Contribution to attention score (pre-softmax) for each head",
)

이러한 주기성은 vocabulary basis가 작업을 수행하기에 가장 자연스러운 basis가 아닐 수도 있다는 생각을 들게 합니다. 문제는 과연 어떤 basis가 적절한가 하는 점입니다.

## Fourier Transforms

> 요약:
>
> * $p$의 약수인 주기를 가진(즉, $2 \pi / p$의 배수인 주파수를 가진) 코사인 및 사인 파형의 Fourier basis를 정의할 수 있습니다.
> * vocab 공간에서 Fourier basis로의 기저 변환(change of basis)을 적용할 수 있으며, 주기 함수는 Fourier basis에서 sparse하게 나타납니다.
> * 하나의 입력에만 의존하는 activation의 경우 1D Fourier transform을 사용하고, 두 입력 모두에 의존하는 activation의 경우 2D Fourier transform을 사용합니다.

현재 어떤 일이 일어나고 있는지 이해하는 자연스러운 방법은 Fourier transforms를 사용하는 것입니다. 이는 모든 함수를 사인 및 코사인 파형의 합으로 표현합니다. 여기서 모든 것은 이산적(discrete)이며, 이는 우리의 함수가 단순히 $p$ 또는 $p^2$ 차원의 벡터라는 것을 의미하고, Fourier transform은 단순히 기저 변환일 뿐입니다. 모든 함수는 Fourier basis에서 *어떤* 형태로든 표현될 수 있지만, "이 함수는 주기적으로 보인다"는 것은 "이 함수는 Fourier basis에서 sparse하다"는 것으로 정의될 수 있습니다.

우리는 원-핫 인코딩된 입력 벡터의 vocabulary 공간에 해당하는 $\mathbb{R}^p$에 기저 변환을 적용하고 있다는 점에 유의하십시오. (우리는 `=`가 vocabulary에 없다고 가정함으로써 표기법을 약간 남용하고 있으며, 따라서 $d_\mathrm{vocab} = p$이 되어 입력 공간에 대해 Fourier transforms를 수행할 수 있게 됩니다.)

### 1D Fourier Basis

1D Fourier basis를 사인 및 코사인 파형의 리스트로 정의합니다. **먼저 상수 파형(constant wave)으로 시작하여, 이후 서로 다른 주파수의 코사인 및 사인 파형을 추가합니다.** 파형은 주기가 $p$의 약수여야 하므로, 주파수는 $\omega_1 = 2 \pi / p $의 정수 배가 됩니다. 우리는 $\vec{\textbf{x}} = (0, 1, ..., (p-1))$라는 약식 표기법을 사용할 것이며, 따라서 $\cos (\omega_k \vec{\textbf{x}})$은 실제로는 $\mathbb{R}^p$의 다음 벡터를 의미합니다:

$$
\cos (\omega_k \vec{\textbf{x}}) = \big(1,\; \cos (\omega_k),\; \cos (2 \omega_k),\; ...,\; \cos ((p-1) \omega_k\big)
$$

(단위 노름으로 스케일링된 후), 여기서 $\omega_k = 2 \pi k / p$입니다. 또한 $F$를 각 **행(row)**이 이러한 파형 하나인 $p \times p$ 행렬로 나타내겠습니다:

$$
F = \begin{bmatrix}
\leftarrow \vec{\textbf{1}} \rightarrow \\
\leftarrow \sin (\omega_1 \vec{\textbf{x}}) \rightarrow \\
\leftarrow \cos (\omega_1 \vec{\textbf{x}}) \rightarrow \\
\leftarrow \sin (\omega_2 \vec{\textbf{x}}) \rightarrow \\
\vdots \\
\leftarrow \cos (\omega_{(p-1)/2} \vec{\textbf{x}}) \rightarrow \\
\end{bmatrix}
$$

마찬가지로 정규화 상수는 생략했지만, 각 행은 노름이 1인 basis vector라고 가정해야 합니다. 이는 상수항 $\vec{\textbf{1}}$은 $\sqrt{\frac{1}{p}}$로 스케일링되고, 나머지는 $\sqrt{\frac{2}{p}}$로 스케일링됨을 의미합니다.

또한 파형(특히 고주파수에서)이 매끄럽지 않고 톱니 모양으로 보인다는 점에 유의하십시오. 이는 우리가 입력을 모든 실수가 아닌 정수로 이산화했기 때문입니다.

### 연습 문제 - 1D Fourier basis 생성하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-25 minutes on this exercise.
> We will be working with the Fourier basis extensively, so it's important to understand what it is.
> ```

아래 함수를 완성하십시오. 계산 효율성에 대해서는 걱정하지 않으셔도 됩니다. for 루프를 사용해도 괜찮습니다.

In [ ]:
def make_fourier_basis(p: int) -> tuple[Tensor, list[str]]:
    """
    Returns a pair `fourier_basis, fourier_basis_names`, where `fourier_basis` is
    a `(p, p)` tensor whose rows are Fourier components and `fourier_basis_names`
    is a list of length `p` containing the names of the Fourier components (e.g.
    `["const", "cos 1", "sin 1", ...]`). You may assume that `p` is odd.
    """
    raise NotImplementedError()


tests.test_make_fourier_basis(make_fourier_basis)

<details><summary>솔루션</summary>

```python
def make_fourier_basis(p: int) -> tuple[Tensor, list[str]]:
    """
    Returns a pair `fourier_basis, fourier_basis_names`, where `fourier_basis` is
    a `(p, p)` tensor whose rows are Fourier components and `fourier_basis_names`
    is a list of length `p` containing the names of the Fourier components (e.g.
    `["const", "cos 1", "sin 1", ...]`). You may assume that `p` is odd.
    """
    # Define a grid for the Fourier basis vecs (we'll normalize them all at the end)
    # Note, the first vector is just the constant wave
    fourier_basis = t.ones(p, p)
    fourier_basis_names = ["Const"]
    for i in range(1, p // 2 + 1):
        # Define each of the cos and sin terms
        fourier_basis[2 * i - 1] = t.cos(2 * t.pi * t.arange(p) * i / p)
        fourier_basis[2 * i] = t.sin(2 * t.pi * t.arange(p) * i / p)
        fourier_basis_names.extend([f"cos {i}", f"sin {i}"])
    # Normalize vectors, and return them
    fourier_basis /= fourier_basis.norm(dim=1, keepdim=True)
    return fourier_basis.to(device), fourier_basis_names
```
</details>

이 작업을 완료하고 (테스트를 통과했다면), 아래 셀을 실행하여 Fourier components를 시각화할 수 있습니다.

In [ ]:
fourier_basis, fourier_basis_names = make_fourier_basis(p)

utils.animate_lines(
    fourier_basis,
    snapshot_index=fourier_basis_names,
    snapshot="Fourier Component",
    title="Graphs of Fourier Components (Use Slider)",
)

*참고 - 이 시점부터 `fourier_basis` 및 `fourier_basis_names` 변수는 전역 변수이므로, 다른 함수에서도 이를 사용하게 됩니다. `p`의 값은 변경하지 않을 것이며, 이 또한 전역 변수입니다.*

이제 임의의 두 벡터의 내적이 동일한 벡터일 때는 1이고, 그렇지 않을 때는 0임을 보여줌으로써 fourier basis가 orthonormal하다는 것을 증명할 수 있습니다. 직접 확인하시려면 다음 셀을 실행하십시오:

In [ ]:
utils.imshow(fourier_basis @ fourier_basis.T, title="Fourier Basis Cosine Similarity Matrix")

Fourier transform가 실제로 orthonormal basis임을 확인했으므로, 이제 모든 $p$-차원 벡터를 이 basis로 표현할 수 있습니다. **1D Fourier transform**은 단순히 standard basis에서의 벡터 성분을 Fourier basis에서의 성분으로 변환하는 과정입니다 (다시 말해, 벡터를 각 Fourier basis 벡터 방향으로 project하는 것입니다).

### 연습 문제 - 1D Fourier transform

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> This should be a short, one-line function.
> Again, this is much more important to understand conceptually as opposed to being difficult to implement.
> ```

이제 벡터의 Fourier transform을 계산하는 함수를 작성해야 합니다. `fourier_basis`의 **행(rows)**들이 Fourier basis 벡터라는 점을 기억하십시오.

In [ ]:
def fft1d(x: Tensor) -> Tensor:
    """
    Returns the 1D Fourier transform of `x`, which can be a vector or a batch of vectors.

    x.shape = (..., p)
    """
    raise NotImplementedError()


tests.test_fft1d(fft1d)

<details>
<summary>솔루션</summary>

```python
def fft1d(x: Tensor) -> Tensor:
    """
    Returns the 1D Fourier transform of `x`, which can be a vector or a batch of vectors.

    x.shape = (..., p)
    """
    return x @ fourier_basis.T
```

참고 - 만약 `x`가 벡터였다면, `fourier_basis @ x`를 반환하는 것이 완전히 괜찮았을 것입니다. 하지만 `x`이 벡터들의 batch라면, 곱셈이 `x`의 마지막 차원을 따라 발생하는지 확인해야 합니다.

</details>

주기적으로 보이는 예시 함수를 통해 이 변환을 입증할 수 있습니다. 핵심 직관은 **'원래 basis에서 함수가 주기적으로 보인다'**는 것이 **'Fourier basis에서 함수가 sparse하다'**는 것을 의미한다는 점입니다. 정수 상의 함수 $[0, p-1]$ 는 $\mathbb{R}^p$ 의 벡터와 동일하다는 점에 유의하십시오. 왜냐하면 그러한 임의의 함수 $f$ 를 다음과 같은 벡터와 연관 지을 수 있기 때문입니다:

$$
\begin{bmatrix}
f(0) \\
f(1) \\
\vdots \\
f(p-1)
\end{bmatrix}
$$

In [ ]:
v = sum([fourier_basis[4], fourier_basis[15] / 5, fourier_basis[67] / 10])

utils.line(v, xaxis="Vocab basis", title="Example periodic function")
utils.line(
    fft1d(v),
    xaxis="Fourier Basis",
    title="Fourier Transform of example function",
    hover=fourier_basis_names,
)

첫 번째 플롯에서는 들쭉날쭉하지만 대략적으로 주기적인 함수를, 두 번째 플롯에서는 (0이 아닌 계수가 단 세 개뿐인) 매우 희소한 함수를 관찰할 수 있습니다.

### 2D Fourier Basis

**위의 모든 아이디어는 $\mathbb{R}^{p \times p}$ 상의 2D Fourier basis, 즉 $p \times p$ 이미지로 자연스럽게 확장될 수 있습니다. 2D Fourier basis의 각 항은 두 개의 1D Fourier basis 항 $v, w$ 의 outer product $v w^T$ 입니다.**

따라서 우리의 2D Fourier basis는 (스케일링 인자를 제외하고) 다음을 포함합니다:

* 상수항 $\vec{\textbf{1}}$,
* $\,\cos(\omega_k \vec{\textbf{x}}),\,\sin(\omega_k \vec{\textbf{x}}),\,\cos(\omega_k \vec{\textbf{y}})$ 및 $\sin(\omega_k \vec{\textbf{y}})$ 형태의 linear 항,
* 그리고 다음과 같은 quadratic 항:

$$
\begin{aligned}
& \cos(w_i\vec{\textbf{x}})\cos(w_j\vec{\textbf{y}}) \\
& \sin(w_i\vec{\textbf{x}})\cos(w_j\vec{\textbf{y}}) \\
& \cos(w_i\vec{\textbf{x}})\sin(w_j\vec{\textbf{y}}) \\
& \sin(w_i\vec{\textbf{x}})\sin(w_j\vec{\textbf{y}})
\end{aligned}
$$

이것들을 길이가 $p^2$ 인 벡터로 생각할 수도 있지만, 크기가 $(p, p)$ 인 행렬로 생각하는 것이 훨씬 더 타당합니다.

> 표기법 - $\cos(\omega_i \vec{\textbf{x}})\cos(\omega_j \vec{\textbf{y}})$ 은 1D 벡터 $\cos (\omega_i \vec{\textbf{x}})$ 와 $\cos (\omega_j \vec{\textbf{y}})$ 의 outer product로 구성된 $(p, p)$ 크기의 행렬로 이해해야 합니다. 다시 말해, 이 행렬의 $(x, y)$ 번째 요소는 $\cos(\omega_i x) \cos(\omega_j y)$ 입니다.

### 연습 문제 - 2D Fourier basis 생성하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> This should be a short, one-line function. Again, this is much more important to understand conceptually.
> ```

다음 함수를 완성하십시오. (1D Fourier basis를 위해 작성했던 함수와 달리) 이 함수는 전체 basis가 아니라 단일 basis term만을 반환한다는 점에 유의하시기 바랍니다.

In [ ]:
def fourier_2d_basis_term(i: int, j: int) -> Float[Tensor, "p p"]:
    """
    Returns the 2D Fourier basis term corresponding to the outer product of the `i`-th component of
    the 1D Fourier basis in the `x` direction and the `j`-th component of the 1D Fourier basis in
    the `y` direction.

    Returns a 2D tensor of length `(p, p)`.
    """
    raise NotImplementedError()


tests.test_fourier_2d_basis_term(fourier_2d_basis_term)

<details>
<summary>솔루션</summary>

```python
def fourier_2d_basis_term(i: int, j: int) -> Float[Tensor, "p p"]:
    """
    Returns the 2D Fourier basis term corresponding to the outer product of the `i`-th component of
    the 1D Fourier basis in the `x` direction and the `j`-th component of the 1D Fourier basis in
    the `y` direction.

    Returns a 2D tensor of length `(p, p)`.
    """
    return fourier_basis[i][:, None] * fourier_basis[j][None, :]
```

`None`를 사용한 인덱싱은 이 함수를 작성하는 여러 방법 중 하나입니다. 다른 몇 가지 방법은 다음과 같습니다:

* `torch.outer`
* einsum 사용: `torch.einsum('i,j->ij', ...)`
* 벡터를 서로 곱하기 전에 `unsqueeze` 메서드를 사용하여 더미 차원을 추가하는 방법입니다.

</details>

이 함수를 정의한 후, 다음 코드를 실행하여 2D Fourier basis를 시각화할 수 있습니다. 이것들이 실제로 주기적으로 보이는지 확인하십시오.

In [ ]:
x_term = 4
y_term = 6

utils.inputs_heatmap(
    fourier_2d_basis_term(x_term, y_term).T,
    title=f"2D Fourier Basis term {fourier_basis_names[x_term]}x {fourier_basis_names[y_term]}y",
)

$(p, p)$ 이미지로 생각하면 어떤 이점이 있을까요? 우리가 다루는 모든 데이터의 batch dimension 크기는 $p^2$ 입니다. 왜냐하면 입력값 `x`와 `y`의 가능한 모든 값을 다루고 있기 때문입니다. 따라서 이 batch dimension을 $(p, p)$로 reshape한 다음, 여기에 2D Fourier transform을 적용하는 방법을 생각할 수 있습니다.

이제 이 transform을 구현해 보겠습니다!

### 연습 문제 - 2D Fourier Transform 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

이미 1D에서 이 작업을 수행해 보셨기 때문에, 이 연습 문제는 매우 익숙하실 것입니다.

In [ ]:
def fft2d(tensor: Tensor) -> Tensor:
    """
    Retuns the components of `tensor` in the 2D Fourier basis.

    Asumes that the input has shape `(p, p, ...)`, where the
    last dimensions (if present) are the batch dims.
    Output has the same shape as the input.
    """
    raise NotImplementedError()


tests.test_fft2d(fft2d)

<details><summary>솔루션</summary>

```python
def fft2d(tensor: Tensor) -> Tensor:
    """
    Retuns the components of `tensor` in the 2D Fourier basis.

    Asumes that the input has shape `(p, p, ...)`, where the
    last dimensions (if present) are the batch dims.
    Output has the same shape as the input.
    """
    # fourier_basis[i] is the i-th basis vector, which we want to multiply along
    return einops.einsum(tensor, fourier_basis, fourier_basis, "px py ..., i px, j py -> i j ...")
```
</details>

1D Fourier transform를 다루면서, 우리는 Fourier basis 벡터들의 선형 결합인 간단한 주기 함수들을 정의했고, 이를 Fourier basis로 표현했을 때 sparse하다는 것을 확인했습니다. 여기에서도 정확히 같은 작업을 수행하지만, 입력이 1개가 아닌 2개인 함수를 대상으로 합니다.

아래는 간단한 2D 주기 함수(2D Fourier basis 항들의 선형 결합)를 그리기 위한 코드입니다. 우리가 이 행렬을 `example_fn`라고 부르는 이유는, 이를 두 개의 입력(x 및 y 방향)에 대한 함수로 생각하기 때문입니다.

In [ ]:
example_fn = sum(
    [
        fourier_2d_basis_term(4, 6),
        fourier_2d_basis_term(14, 46) / 3,
        fourier_2d_basis_term(97, 100) / 6,
    ]
)

utils.inputs_heatmap(example_fn.T, title="Example periodic function")

이 함수가 2D Fourier basis에서 sparse하다는 것을 보여주는 코드입니다 (0이 아닌 계수들을 확인하려면 확대해야 합니다):

In [ ]:
utils.imshow_fourier(fft2d(example_fn), title="Example periodic function in 2D Fourier basis")

이 코드를 실행하여, 0이 아닌 성분들이 함수를 구성하는 데 사용한 basis 항들과 정확히 일치하는지 확인하실 수 있습니다.

## Fourier Transform를 이용한 모델 분석

지금까지 우리는 두 가지 관찰 결과를 얻었습니다:

* 모델 activation의 상당수가 주기적으로 보입니다.
* 주기 함수는 Fourier basis에서 sparse하게 나타납니다.

따라서 당연한 다음 단계로, activation에 2D Fourier transformation을 적용해 보겠습니다! activation의 batch 차원은 $p^2$이며, 이를 $(p, p)$로 재배열할 수 있다는 점을 기억하십시오. 이 두 차원은 모듈러 산술 방정식의 `x` 및 `y` 입력을 나타냅니다. 우리가 Fourier transform을 수행할 차원이 바로 이 부분입니다.

### Fourier basis에서 activation 시각화하기

각 head에 대해 token 2가 token 0에 주는 attention score의 heatmap을 그리기 위해 작성했던 이전 코드를 상기해 보겠습니다:

```python
inputs_heatmap(
    attn_mat[..., 0],
    title=f'Attention score for heads at position 0',
    animation_frame=2,
    animation_name='head'
)
```

해당 plot에서 x축과 y축은 모듈러 산술 방정식의 입력 `x`와 `y`의 서로 다른 값들을 나타냈습니다.

아래 코드는 attention matrix의 2D Fourier transform을 수행하며, 각 head에 대해 token 2가 token 0에 주는 attention score의 heatmap을 Fourier basis에서 시각화합니다:

In [ ]:
# Apply Fourier transformation
attn_mat_fourier_basis = fft2d(attn_mat_sq)

# Plot results
utils.imshow_fourier(
    attn_mat_fourier_basis[..., 0],
    title="Attention score for heads at position 0, in Fourier basis",
    animation_frame=2,
    animation_name="head",
)

결과가 매우 희소(sparse)하다는 것을 알 수 있을 것입니다. 0이 아닌 셀은 아주 적으며, 대부분 0번째 행이나 열(즉, 상수항 또는 선형항에 해당)에 위치합니다. 이는 우리가 2D Fourier basis를 사용하는 올바른 방향으로 가고 있음을 시사합니다!

이제 neuron activation에 대해서도 동일한 작업을 수행하겠습니다. 이전 코드를 다시 살펴보겠습니다:

```python
top_k = 3
inputs_heatmap(
    neuron_acts_post[:, :top_k],
    title=f'Activations for first {top_k} neurons',
    animation_frame=2,
    animation_name='Neuron'
)
```

여기서도 정확히 동일한 작업을 수행하고, Fourier basis에서 activation을 시각화하겠습니다:

In [ ]:
neuron_acts_post_fourier_basis = fft2d(neuron_acts_post_sq)

top_k = 3
utils.imshow_fourier(
    neuron_acts_post_fourier_basis[..., :top_k],
    title=f"Activations for first {top_k} neurons",
    animation_frame=2,
    animation_name="Neuron",
)

### 연습 문제 - activation에서 패턴 찾기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

`top_k`을 3에서 더 큰 숫자로 늘리고, 서로 다른 neuron들을 살펴보십시오. activation의 패턴에서 무엇이 느껴지십니까?

<details>
<summary>정답 (보여야 할 내용)</summary>

마찬가지로 sparsity가 관찰되어야 하며, 이번에는 일부 quadratic 항들도 보일 것입니다.

그 외에도 언급할 만한 두 가지 뚜렷한 패턴이 있습니다:

* 각 neuron은 동일한 non-zero 항 패턴을 가집니다: $k$의 특정 값에 대해, non-zero 항들은 상수항과 frequency $k$만을 포함하는 4개의 linear 항 및 4개의 quadratic 항(즉, $\cos(\omega_k \vec{\textbf{x}})$, $\sin(\omega_k \vec{\textbf{y}})$, $\cos(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}})$ 등과 같은 항들)으로 구성됩니다.
* 모든 neuron 전체에서 $k$의 서로 다른 값은 소수에 불과하며, 따라서 많은 neuron이 매우 유사해 보이는 activation 패턴을 갖게 됩니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/activation-patterns.png" width="650">
</details>

#### 참고: batch 차원의 기저 변환 (Change of basis)

batch 차원에서 기저 변환을 수행하는 것은 꽤 생소한 작업이며, 여기서 어떤 일이 일어나는지 신중하게 생각할 가치가 있습니다 (이는 이전의 Transformer Circuits 연구와 크게 다른 점이며, 전체 유니버스를 하나의 batch로 넣을 수 있을 만큼 매우 작은 toy problem이기 때문에 여기서만 유효한 접근 방식이라는 점에 유의하십시오).

batch 차원에 대해 선형적이지 않은 연산이 *네 가지* 있습니다. 앞서 언급한 attention softmax, ReLU, 그리고 최종 softmax가 그것입니다. 또한 attention pattern과의 elementwise multiplication 역시 마찬가지입니다.

특히, ReLU는 매우 특이하게 변합니다. 단순한 elementwise 연산에서 'Fourier 기저의 역행렬로 회전시키고, *새로운* 기저에서 elementwise로 ReLU를 적용한 뒤, 다시 Fourier 기저로 회전시키는' 연산이 됩니다.

### Fourier 기저에서 effective weights 시각화하기

activation을 시각화하는 것 외에도, weight 행렬을 직접 살펴볼 수 있습니다.

*참고 - 이 섹션은 노트북의 나머지 부분을 이해하는 데 필수적이지 않으므로, 시간이 부족하시다면 자유롭게 건너뛰셔도 좋습니다.*

이제 표준 기저에서 `W_neur` 행렬을 시각화했던 이전 코드를 수정하여, Fourier 기저에서 시각화되도록 하겠습니다. 아래 플롯에서 각 선은 입력 token `x`의 함수로서 특정 neuron의 activation을 보여줍니다 (만약 위치 2에 있는 `=` token이 token `x`에만 attention을 기울였다면). neuron 인덱스는 슬라이더 값에 의해 결정됩니다. 이 부분이 다소 혼란스럽다면, 아래 드롭다운을 사용하여 이 transformer의 functional form과 $W_{neur}$의 역할을 다시 확인하실 수 있습니다.

<details>
<summary>Functional Form</summary>

$$
f(t)=\operatorname{ReLU}\Bigg(\sum_h{\bigg(\alpha^h t_0\;+\;\left(1\;-\;\alpha^h\right) t_1 \bigg)^T}{W_{neur}^h}\Bigg) \; W_{logit}
$$

이를 통해 $W_{neur}^h$의 역할을 명확하게 알 수 있습니다. 이는 $(d_{vocab}, d_{mlp})$ 형태의 행렬이며, 각 행은 MLP activation(pre-ReLU)을 얻기 위해 가중 평균을 내는 벡터들입니다.
</details>

아래 코드는 동일한 플롯을 생성하지만, 이전 플롯이 표준 기저(x축이 입력 token을 나타냄)였던 것과 달리, 이 플롯의 x축은 각 Fourier 기저 방향에 대한 입력 token의 성분입니다.

특정 차원에 대해 1D Fourier transform을 수행하는 헬퍼 함수 `fft1d_given_dim`를 제공해 드렸습니다. `W_neur`은 `(n_heads, d_vocab, d_mlp)` 형태이며, 우리는 `d_vocab` 차원에 대해 변환을 수행하려 하므로 이 함수가 필요합니다.

In [ ]:
def fft1d_given_dim(tensor: Tensor, dim: int) -> Tensor:
    """
    Performs 1D FFT along the given dimension (not necessarily the last one).
    """
    return fft1d(tensor.transpose(dim, -1)).transpose(dim, -1)


W_neur_fourier = fft1d_given_dim(W_neur, dim=1)

top_k = 5
utils.animate_multi_lines(
    W_neur_fourier[..., :top_k],
    y_index=[f"head {hi}" for hi in range(4)],
    labels={"x": "Fourier component", "value": "Contribution to neuron"},
    snapshot="Neuron",
    hover=fourier_basis_names,
    title=f"Contribution to first {top_k} neurons via OV-circuit of heads (not weighted by attn), in Fourier basis",
)

각 라인 플롯은 일반적으로 하나만 0이 아닌 것이 아니라, $\sin k$ 및 $\cos k$ 항이 모두 0이 아니라는 점에 유의하시기 바랍니다.

마지막으로, `W_attn`에 대해서도 동일한 작업을 수행하겠습니다:

In [ ]:
utils.lines(
    fft1d(W_attn),
    labels=[f"Head {hi}" for hi in range(4)],
    xaxis="Input token",
    yaxis="Contribution to attn score",
    title="Contribution to attn score (pre-softmax) for each head, in Fourier Basis",
    hover=fourier_basis_names,
)

마지막 두 선 그래프에서 0이 아닌 소수의 주파수들이 attention pattern에서 읽어낸 중요한 주파수들과 정확히 일치한다는 점을 눈치채셨을 것입니다!

## 섹션 요약

이 섹션에서 배운 내용을 복습해 보겠습니다. 우리는 다음과 같은 사실을 발견했습니다:

> - 1-layer 모델의 단순한 구조는 학습된 모든 솔루션의 함수 형태를 크게 제한합니다.
>     - 특히, (몇 가지 단순화된 가정을 한 후에는) 모델의 동작을 완전히 설명하는 몇 개의 행렬을 정의할 수 있습니다.
> - 모델의 내부 activation 중 상당수가 입력에 대해 주기적인 특성을 보이는 것으로 나타났습니다 `x`, `y` (예: attention pattern 및 neuron activation).
> - 주기 함수를 표현하는 자연스러운 방법은 Fourier basis를 사용하는 것입니다. 주기 함수는 이 basis에서 sparse하게 나타납니다.
> - 이는 우리 모델이 단 몇 개의 주파수만을 사용하고(즉, 입력을 몇 개의 서로 다른 Fourier basis 벡터로 투영하고), 나머지는 버리고 있을 가능성을 시사합니다.
> - 우리는 다음 사항들을 살펴봄으로써 이 가설을 확인했습니다:
>     - 모델의 activation (즉, attention pattern 및 neuron activation)
>     - 모델의 effective weight matrices (즉, $W_{attn}$ 및 $W_{neur}$)
> - 이 두 가지 관찰 모두 Fourier basis에서 sparsity가 존재함을 확인해 주었습니다. 더욱이, 모든 경우에 동일한 소수의 주파수들이 나타나는 것으로 보였습니다.

# 2️⃣ Circuit 및 Feature 분석

> ##### 학습 목표
>
> * 1D 및 2D Fourier basis에 대한 이해를 적용하여, 모델의 activation / effective weight가 Fourier basis에서 매우 sparse하다는 것을 보여줍니다.
> * 이러한 관찰 결과를 모델 알고리즘에 대한 구체적인 가설로 전환합니다.
> * 통계적 방법과 ablation과 같은 intervention을 사용하여 이러한 가설들을 검증합니다.
> * 모델의 알고리즘과 모델이 태스크를 해결하는 방법을 완전히 이해합니다.

transformer를 이해하는 과정은 크게 두 가지 부분으로 나뉩니다. 하나는 비선형 activation(output probabilities, attention patterns, neuron activations)으로 표현되는 feature를 해석하는 것이고, 다른 하나는 각 feature를 계산하는 circuit(이전 feature를 이후 feature로 변환하기 위해 수행되는 계산을 나타내는 weight)을 해석하는 것입니다. 이 섹션에서는 embedding, neuron activations(feature), 그리고 logit 계산(circuit)을 해석합니다. 이들이 해석해야 할 가장 중요한 부분들입니다.

embedding부터 시작하겠습니다.

## embedding 이해하기

아래는 Fourier basis에서 embedding을 시각화하기 위한 코드입니다. 이 코드를 실행하고 출력 결과를 해석하시기 바랍니다.

In [ ]:
utils.line(
    (fourier_basis @ W_E).pow(2).sum(1),
    hover=fourier_basis_names,
    title="Norm of embedding of each Fourier Component",
    xaxis="Fourier Component",
    yaxis="Norm",
)

<details>
<summary>해석</summary>

embedding이 Fourier basis에서 sparse하며, 소수의 주파수를 제외한 모든 Fourier component를 버린다는 것을 알 수 있습니다 (주파수의 개수와 값은 임의적이며, 학습 실행마다 달라집니다).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/fourier_norms.png" width="500">

component $2k$에 대한 Fourier basis vector는 다음과 같습니다:

$$
\cos\left(\frac{2 \pi k}{p}x\right)_{x = 0, ..., p-1}
$$

$2k-1$의 경우도 동일하지만, $\cos$이 $\sin$로 대체됩니다.

따라서 이 결과는 입력 `x`에 대해, 주요 주파수 $k$ 각각에 대한 정보 $\cos\left(\frac{2 \pi k}{p}x\right)$는 유지하고, 다른 모든 주파수 $\omega$에 대한 정보는 버리고 있음을 알려줍니다.

norm이 유의미한 주파수를 **key frequencies**라고 부르겠습니다 (여기서는 14, 31, 35, 41, 42, 52입니다).

</details>

<details>
<summary>또 다른 관점 (singular value decomposition으로부터)</summary>

모든 행렬은 orthogonal matrix, diagonal matrix, 그리고 또 다른 orthogonal matrix의 곱으로 나타낼 수 있음을 상기하십시오:

$$
\begin{aligned}
A &= U S V^T \\
    &= \sum_{i=1}^k \sigma_i u_i v_i^T
\end{aligned}
$$

여기서 $u_i$, $v_i$는 orthogonal matrix $U$, $V$의 column vector들이며, $\sigma_1, ..., \sigma_k$는 $A$의 0이 아닌 singular value들입니다. 이 개념이 익숙하지 않다면, SVD를 더 자세히 다루는 induction heads 연습 문제를 살펴보시기 바랍니다. 이는 low-rank matrix를 표현하는 자연스러운 방법인 경우가 많습니다 (대부분의 singular value $\sigma_i$가 0이 되기 때문입니다).

행렬 `fourier_basis`를 $F$로 표시하겠습니다 (행들이 basis vector임을 기억하십시오). 여기서 우리는 행렬 $F W_E$이 매우 sparse하다는 것(즉, 대부분의 row vector가 0이라는 것)을 발견했습니다. 이를 통해 $W_E$이 다음과 같은 low-rank SVD로 잘 근사될 수 있음을 추론할 수 있습니다:

$$
W_E \approx F^T S V^T
$$

왜냐하면 $F$를 왼쪽에 곱하면, 0이 아닌 singular value에 해당하는 행을 제외하고 모든 곳이 0인 행렬이 되기 때문입니다:

$$
F W_E \approx F^T F S V^T = S V^T
$$

다시 말해, token $t_0$, $t_1$의 embedding을 취할 때 우리가 투영하는 **input directions**는 대략적으로 Fourier basis directions와 같습니다.

행렬 곱 $S V^T$ ($S$의 대부분의 diagonal value가 0이므로 행 방향으로 sparse할 것입니다)를 직접 시각화하려면 다음을 실행하십시오:

```python
imshow_div(fourier_basis @ W_E)
```
</details>

## neuron activation 이해하기

이제 $W_E$이 몇 가지 핵심 주파수에 해당하는 정보만 보존하고 있다는 점을 확인했으므로, 이러한 주파수들이 실제로 어디에 사용되는지 살펴보겠습니다.

> 요약:
> * 각 neuron의 activation은 상수항과 **특정 주파수의 선형 및 이차항**의 선형 결합입니다.
> * neuron들은 핵심 주파수에 따라 명확하게 **클러스터링됩니다**.

### Neuron은 quadratic term을 생성합니다

먼저, 이전 섹션에서 작성했던 input basis와 2D Fourier basis에서의 neuron activation 다이어그램을 다시 살펴보겠습니다:

```python
top_k = 5
utils.inputs_heatmap(neuron_acts_post_sq[..., :top_k], ...)
utils.imshow_fourier(neuron_acts_post_fourier_basis[..., :top_k], ...)
```

첫 번째 플롯은 주기적인 형태(표준 basis에서 주기적임 = Fourier basis에서 sparse함)를 띠고 있으며, 두 번째 플롯은 **각 neuron이 이전에 관찰했던 주요 주파수 중 일부와 연관되어 있음**을 보여줌으로써 Fourier basis 표현에 대한 더 자세한 정보를 제공한다는 것을 확인했습니다.

예를 들어, 이 플롯들의 첫 번째 neuron을 살펴보십시오. 2D Fourier basis에서 중요한 주파수는 상수항과 $\omega = 42$에 해당하는 주파수뿐임을 알 수 있습니다 ($\sin$와 $\cos$ 항이 모두 나타납니다). 결과적으로 이는 (스케일 인자를 제외하고) 총 9개의 항을 제공합니다:

$$
\begin{bmatrix}
1 & \cos(\omega_k x) & \sin(\omega_k x) \\
\cos(\omega_k y) & \cos(\omega_k x)\cos(\omega_k y) & \sin(\omega_k x)\cos(\omega_k y) \\
\sin(\omega_k y) & \cos(\omega_k x)\sin(\omega_k y) & \sin(\omega_k x)\sin(\omega_k y)
\end{bmatrix}
$$

여기서 $\omega_k = 2 \pi k / p$이며, 이 경우에는 $k = 42$입니다. 여기에는 상수항, 4개의 linear term, 그리고 4개의 quadratic term이 포함됩니다.

이것의 의미는 무엇일까요? 중요한 점은 다음과 같은 삼각함수 공식이 있다는 것입니다:

$$
\begin{aligned}
\cos(\omega_k(x + y)) = \cos(\omega_k x) \cos(\omega_k y) - \sin(\omega_k x) \sin(\omega_k y) \\
\sin(\omega_k(x + y)) = \sin(\omega_k x) \cos(\omega_k y) + \cos(\omega_k x) \sin(\omega_k y)
\end{aligned}
$$

플롯들은 이 방정식들의 우변에 있는 항들(즉, quadratic term들)이 우리 neuron에 의해 감지되고 있음을 알려줍니다. 모델이 modular addition을 수행하기 위해 결국 내부적으로 $x+y$이라는 양을 어떤 방식으로든 표현해야 한다는 것을 알고 있으므로, 모델이 이러한 방식(즉, 우변의 값들을 먼저 계산함으로써 좌변의 양을 계산하는 방식)으로 이를 수행한다고 추측할 수 있습니다. 다시 말해, 우리 neuron들은 어떤 의미에서 서로 다른 주파수 $k$에 대해 $\cos(\omega_k(x + y))$와 $\sin(\omega_k(x + y))$ 정보를 저장하고 있는 것입니다.

이제 이러한 2D Fourier basis 항들의 계수 일부를 더 자세히 살펴보겠습니다.

### 연습 문제 - 평균 제곱 계수 계산하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> This exercise asks you to perform some basic operations and make some simple plots with your neuron activations.
> ```

우리는 뉴런 activation(입력 $x$, $y$의 함수입니다)이 어떤 의미에서 다음과 같은 함수 $\sum_i \sum_j B_{i, j} F_{i, x} F_{j, y}$를 근사하려고 시도한다고 추측할 수 있습니다:

* $F$은 Fourier 기저 변환 행렬입니다.
    * 예를 들어, $F_{1,17}= \cos(\frac{2\pi}{p} 17)$ 및 $F_{2, 17} = \sin(\frac{2\pi}{p} 17)$ 임에 유의하십시오.
* $B$은 계수 행렬이며, 우리는 이것이 매우 sparse할 것이라고 추측합니다.
    * 구체적으로, 모델이 근사하려는 "실제 함수"가 위 행렬에 있는 9개 항의 선형 결합과 유사한 형태일 것이라고 추측합니다:

$$
\sum_{i\in \{0, 2\omega-1, 2\omega\}} \sum_{j\in \{0, 2\omega-1, 2\omega\}} B_{i,j} F_{i,x} F_{j,y}
$$

($2\omega-1$번째와 $2\omega$번째 기저 벡터가 주파수 $\omega$에 대한 cosine 및 sine 벡터임을 기억하십시오).

각 뉴런에 대한 평균 제곱 계수의 heatmap을 생성하십시오 (다시 말해, heatmap의 `(i, j)`번째 값은 모든 뉴런에 걸친 $B_{i, j}^2$의 평균입니다).

코드는 다음 두 단계로 구성되어야 합니다:

* 모든 batch에 대한 평균을 빼서 뉴런 activation을 centering 합니다 (이는 기본적으로 bias 항을 제거하기 때문입니다). 모든 ReLU 뉴런은 항상 non-negative activation을 가지며, 상수항은 보통 별도로 고려해야 합니다.
* centering된 뉴런 activation의 2D Fourier transform을 수행합니다.

결과를 플롯하기 위한 코드는 이미 제공되었습니다.

batch 차원이 이미 grid 형태로 구성되어 있는 `neuron_acts_post_sq` 객체를 사용하여 작업하는 것을 잊지 마십시오.

In [ ]:
# YOUR CODE HERE - compute neuron activations (centered)
neuron_acts_centered = neuron_acts_post_sq - neuron_acts_post_sq.mean((0, 1), keepdim=True)

# Take 2D Fourier transform
neuron_acts_centered_fourier = fft2d(neuron_acts_centered)


utils.imshow_fourier(
    neuron_acts_centered_fourier.pow(2).mean(-1),
    title="Norms of 2D Fourier components of centered neuron activations",
)

평균 $B_{i, j}^2$ 값의 plot은 이전에 확인했던 것과 같은 sparsity를 보여주어야 합니다. 즉, 소수의 선택된 frequency에 해당하는 const, linear, quadratic 항들만이 0이 아닌 값으로 나타날 것입니다. 이러한 frequency들을 이전에 정의한 **key frequencies**와 비교하여, 그것들이 동일한지 검증할 수 있습니다.

### 이차항(quadratic terms)은 어떻게 얻을까요?

이러한 이차항들이 정확히 어떻게 계산되는지는 다소 복잡합니다. 신경망에 대한 좋은 멘탈 모델은, 신경망이 행렬 곱셈과 덧셈에는 매우 능숙하지만 그 외의 작업에는 많은 노력이 필요하다는 것입니다. 따라서 입력의 서로 다른 두 부분의 곱을 구하는 것은 상당히 어려울 것이라고 예상해야 합니다!

이 계산에 대한 더 자세한 내용은 [original notebook](https://colab.research.google.com/drive/1F6_1_cWXE5M7WocUcpQWp3v8z4b1jL20#scrollTo=ByvnsimvZXGC)에서 확인하실 수 있습니다. 요약하자면, **모델은 ReLU activation과 attention을 통한 element-wise product를 모두 사용하여, 다소 편법적인 방식으로 항들을 곱합니다**.

attention pattern 곱(여기서는 자세히 다루지 않습니다)이 작동하는 이유는 attention이 value 벡터와 attention 확률의 곱을 취하는 과정을 포함하며, 이 각각이 입력의 함수이기 때문입니다. attention 확률과 value 벡터는 보통 서로 다른 역할을 하는 것으로 생각되기 때문에 이것이 다소 이상하게 보일 수 있지만, 여기서는 모델이 입력의 서로 다른 두 부분을 곱할 수 있도록 함께 작동하고 있습니다.

ReLU activation 또한 꽤 놀랍습니다. 1D 및 2D Fourier 성분의 선형 함수는 1D Fourier 성분의 선형 함수에 ReLU를 적용한 것으로 잘 근사된다는 것이 밝혀졌습니다. 구체적으로, 다음 식을 근사하면:

$$
\operatorname{ReLU}(A + B \cos(\omega x) + B \cos(\omega y))
$$

($A$, $B > 0$에 대하여) 이를 2D Fourier basis의 다음 4개 항의 선형 결합으로 나타낼 때:

$$
\alpha + \beta \cos(\omega x) + \beta \cos(\omega y) + \gamma \cos(\omega x) \cos(\omega y)
$$

$\cos(\omega x) \cos(\omega y)$ 방향으로 상당한 성분이 포함되어 있음을 알 수 있습니다. 이러한 현상이 발생하는 핵심 직관은 이차항이 **$x$과 $y$ 사이의 상호작용**을 포착한다는 것입니다. $\operatorname{ReLU}$은 $\cos(\omega x) + \cos(\omega y)$의 [convex function](https://en.wikipedia.org/wiki/Convex_function)이므로, ($x$과 $y$가 확률 변수라고 가정하면) 이 두 입력이 상관관계가 있을 때 기대값이 더 커지며, 따라서 $\gamma > 0$이 됩니다. \*

\* *참고 - 이는 상당히 대략적인 논증이므로, 직관적으로 이해되지 않더라도 너무 걱정하지 마십시오!*

이것이 중요한 이유는 우리 모델이 이 두 식 중 첫 번째 식은 계산할 수 있지만(attention layer에서 $\cos(\omega x)$와 $\cos(\omega y)$의 선형 결합을 취한 다음, MLP 과정에서 $\operatorname{ReLU}$을 적용할 수 있음), 두 번째 식은 직접 계산할 수 없기 때문입니다. 따라서 우리는 기본적으로 ReLU를 사용하여 선형 항과 이차항의 합을 근사할 수 있습니다.

### 연습 문제 - quadratic 항이 중요하며, $\gamma > 0$ 임을 확인하십시오

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> Doing this exercise isn't super valuable to the overall experience of this section. You can skip it if you want.
> ```

$A = {1}/{2\sqrt{p}}, \; B = 1, \;$와 $\;\omega = \omega_{42} = (2 \pi \times 42) / p \;$(앞서 관찰한 핵심 주파수 중 하나)를 사용하십시오. ReLU 근사치와 실제 quadratic 함수 사이의 mean squared error를 최소화하는 계수 $\alpha$, $\beta$ 및 $\gamma$을 찾으십시오. 이 피팅의 $r^2$ 점수를 구하십시오. $\gamma > 0$ 임을 확인하고, 점수가 1에 가까운지 확인하십시오. 또한, quadratic 항을 제외했을 때 $r^2$ 점수가 상당히 감소하는지 확인하십시오 (이는 이 quadratic 항이 중요하다는 것을 보여줍니다).

`sklearn`의 [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) 함수를 사용할 수 있습니다.

*회귀 분석 시 정규화된 1D 및 2D Fourier basis 벡터를 사용하는 것을 잊지 마십시오.*

In [ ]:
from sklearn.linear_model import LinearRegression

# YOUR CODE HERE - compute quadratic term, and r^2 of regression with/without it

<details>
<summary>결과 논의</summary>

다음과 같은 결과를 얻어야 합니다:

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">ReLU(0.5 + cos(wx) + cos(wy)) ≈ 9.190*const + 6.807*cos(wx) + 6.807*cos(wy) + 3.566*cos(wx)cos(wy)
r2: 0.966
r2 (quadratic term 제외): 0.849</pre>

이는 quadratic term이 실제로 양수 계수를 가지며, ReLU 함수의 분산 중 상당 부분을 설명한다는 것을 확인시켜 줍니다 (구체적으로, linear term을 고려한 후 남은 분산의 2/3 이상을 설명합니다).

(quadratic term이 없는 regression의 계수는 출력하지 않았음에 유의하십시오. 우리의 2D Fourier basis 벡터들은 orthogonal하므로, 이 계수들이 quadratic term이 포함된 regression의 계수와 동일함을 보장할 수 있습니다.)

</details>


<details><summary>솔루션</summary>

```python
from sklearn.linear_model import LinearRegression

# Choose a particular frequency, and get the corresponding cosine basis vector
k = 42
idx = 2 * k - 1
vec = fourier_basis[idx]

# Get ReLU function values
relu_func_values = F.relu(0.5 * (p**-0.5) + vec[None, :] + vec[:, None])

# Get terms we'll be using to approximate it
# Note we're including the constant term here
data = t.stack([fourier_2d_basis_term(i, j) for (i, j) in [(0, 0), (idx, 0), (0, idx), (idx, idx)]], dim=-1)

# Reshape, and convert to numpy
data = to_numpy(data.reshape(p * p, 4))
relu_func_values = to_numpy(relu_func_values.flatten())

# Fit a linear model (we don't need intercept because we have const Fourier basis term)
reg = LinearRegression(fit_intercept=False).fit(data, relu_func_values)
coefs = reg.coef_
r2 = reg.score(data, relu_func_values)
print(
    "ReLU(0.5 + cos(wx) + cos(wy)) ≈ {:.3f}*const + {:.3f}*cos(wx) + {:.3f}*cos(wy) + {:.3f}*cos(wx)cos(wy)".format(
        *coefs
    )
)
print(f"r2: {r2:.3f}")

# Run the regression again, but without the quadratic term
data = data[:, :3]
reg = LinearRegression().fit(data, relu_func_values)
coefs = reg.coef_
bias = reg.intercept_
r2 = reg.score(data, relu_func_values)
print(f"r2 (no quadratic term): {r2:.3f}")
```
</details>

### 주파수별 뉴런 클러스터링

각 뉴런이 가장 민감하게 반응하는 단일 주파수가 있고 (나머지는 모두 무시하는 것으로) 보인다는 점을 확인했으므로, 이제 이 주파수에 따라 뉴런들을 정렬해 보고, 각 주파수가 뉴런의 동작을 설명하는 데 얼마나 효과적인지 확인해 보겠습니다.

### 연습 문제 - neuron cluster 찾기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> This exercise is conceptually important, but quite challenging.
> ```

각 neuron에 대해, 해당 frequency를 포함하는 Fourier component들이 그 neuron activation의 variance를 가장 많이 설명하는 frequency를 찾아야 합니다 (다시 말해, 특정 neuron에 대해 해당 frequency의 const, linear, quadratic 항들의 제곱 합이 모든 Fourier coefficient의 제곱 합에서 차지하는 비율이 가장 큰 frequency가 무엇인지 찾는 것입니다).

이를 위해 헬퍼 함수 `arrange_by_2d_freqs`를 제공했습니다. 이 함수는 2D Fourier basis의 coefficient 텐서(shape `(p, p, ...)`)를 입력으로 받아, 각 frequency별로 정렬된 Fourier coefficient를 나타내는 shape `(p//2, 3, 3, ...)`의 텐서를 반환합니다. 즉, 이 텐서의 `[k-1, ...]`번째 slice는 다음과 같은 (정규화된) 2D Fourier basis vector들에 대한 Fourier coefficient를 포함하는 `(3, 3, ...)` shape의 텐서가 됩니다:

$$
\begin{bmatrix}
1 & \cos(\omega_k x) & \sin(\omega_k x) \\
\cos(\omega_k y) & \cos(\omega_k x)\cos(\omega_k y) & \sin(\omega_k x)\cos(\omega_k y) \\
\sin(\omega_k y) & \cos(\omega_k x)\sin(\omega_k y) & \sin(\omega_k x)\sin(\omega_k y)
\end{bmatrix}
$$

이는 단순히 텐서를 정교하게 재배치한 것으로 생각하시면 됩니다. 이를 통해 특정 frequency에 대한 모든 (const, linear, quadratic) 항들에 더 쉽게 접근할 수 있습니다.

In [ ]:
def arrange_by_2d_freqs(tensor: Tensor) -> Tensor:
    """
    Takes a tensor of shape (p, p, *batch_dims), returns tensor of shape (p//2 - 1, 3, 3, *batch_dims)
    representing the Fourier coefficients sorted by frequency (each slice contains const, linear and
    quadratic terms).

    In other words, if `tensor` had shape (p, p) and looked like this:

        1           cos(w_1*x)            sin(w_1*x)           ...
        cos(w_1*y)  cos(w_1*x)cos(w_1*y)  sin(w_1*x)cos(w_1*y) ...
        sin(w_1*y)  cos(w_1*x)sin(w_1*y)  sin(w_1*x)sin(w_1*y) ...
        cos(w_2*y)  cos(w_1*x)cos(w_2*y)  sin(w_1*x)cos(w_2*y) ...
        ...         ...                   ...

    Then arrange_by_2d_freqs(tensor)[k-1] should be the following (3, 3) tensor of kth-mode Fourier
    frequencies:

        1           cos(w_k*x)            sin(w_k*x)
        cos(w_k*y)  cos(w_k*x)cos(w_k*y)  sin(w_k*x)cos(w_k*y)
        sin(w_k*y)  cos(w_k*x)sin(w_k*y)  sin(w_k*x)sin(w_k*y)

    for k = 1, 2, ..., p//2. Note we omit the constant term, i.e. the 0th slice has frequency k=1.

    Any dimensions beyond the first 2 are treated as batch dimensions, i.e. we only rearrange the
    first 2.
    """
    idx_2d_y_all = []
    idx_2d_x_all = []
    for freq in range(1, p // 2):
        idx_1d = [0, 2 * freq - 1, 2 * freq]
        idx_2d_x_all.append([idx_1d for _ in range(3)])
        idx_2d_y_all.append([[i] * 3 for i in idx_1d])
    return tensor[idx_2d_y_all, idx_2d_x_all]


def find_neuron_freqs(
    fourier_neuron_acts: Float[Tensor, "p p d_mlp"],
) -> tuple[Float[Tensor, "d_mlp"], Float[Tensor, "d_mlp"]]:
    """
    Returns the tensors `neuron_freqs` and `neuron_frac_explained`, containing the frequencies that
    explain the most variance of each neuron and the fraction of variance explained, respectively.
    """
    fourier_neuron_acts_by_freq = arrange_by_2d_freqs(fourier_neuron_acts)
    assert fourier_neuron_acts_by_freq.shape == (p // 2 - 1, 3, 3, utils.d_mlp)

    raise NotImplementedError()

    return neuron_freqs, neuron_frac_explained


neuron_freqs, neuron_frac_explained = find_neuron_freqs(neuron_acts_centered_fourier)
key_freqs, neuron_freq_counts = t.unique(neuron_freqs, return_counts=True)

assert key_freqs.tolist() == [14, 35, 41, 42, 52]

print("All tests for `find_neuron_freqs` passed!")

<details>
<summary>도움말 - 모든 key frequency가 정답보다 1씩 차이가 납니다.</summary>

`arrange_by_2d_freqs` 함수의 0번째 slice는 사실 $k=0$가 아니라 $k=1$의 frequency라는 점을 기억하십시오 (이후로도 마찬가지입니다). 만약 argmax를 통해 key frequency를 구했다면, 해당 인덱스에 1을 더해야 합니다!
</details>


<details>
<summary>솔루션</summary>

```python
def find_neuron_freqs(
    fourier_neuron_acts: Float[Tensor, "p p d_mlp"],
) -> tuple[Float[Tensor, "d_mlp"], Float[Tensor, "d_mlp"]]:
    """
    Returns the tensors `neuron_freqs` and `neuron_frac_explained`, containing the frequencies that
    explain the most variance of each neuron and the fraction of variance explained, respectively.
    """
    fourier_neuron_acts_by_freq = arrange_by_2d_freqs(fourier_neuron_acts)
    assert fourier_neuron_acts_by_freq.shape == (p // 2 - 1, 3, 3, utils.d_mlp)

    # Sum squares of all frequency coeffs, for each neuron
    square_of_all_terms = einops.reduce(fourier_neuron_acts.pow(2), "x_coeff y_coeff neuron -> neuron", "sum")

    # Sum squares just corresponding to const+linear+quadratic terms,
    # for each frequency, for each neuron
    square_of_each_freq = einops.reduce(
        fourier_neuron_acts_by_freq.pow(2), "freq x_coeff y_coeff neuron -> freq neuron", "sum"
    )

    # Find the freq explaining most variance for each neuron
    # (and the fraction of variance explained)
    neuron_variance_explained, neuron_freqs = square_of_each_freq.max(0)
    neuron_frac_explained = neuron_variance_explained / square_of_all_terms

    # The actual frequencies count up from k=1, not 0!
    neuron_freqs += 1

    return neuron_freqs, neuron_frac_explained
```

솔루션에서 단순히 `fourier_neuron_acts.pow(2).sum((0, 1))` 등을 사용하는 대신 `einops.reduce`을 사용한 점에 주목하십시오. `einops`이 유용한 대부분의 상황과 마찬가지로, 이 방식은 코드를 더 명시적이고 읽기 쉽게 만들며 실수의 가능성을 줄여준다는 장점이 있습니다.

</details>

이 함수를 작성하고 테스트를 통과했다면, 설명된 분산의 비율(fraction of variance explained)을 그래프로 그릴 수 있습니다.

In [ ]:
fraction_of_activations_positive_at_posn2 = (cache["pre", 0][:, -1] > 0).float().mean(0)

utils.scatter(
    x=neuron_freqs,
    y=neuron_frac_explained,
    xaxis="Neuron frequency",
    yaxis="Frac explained",
    colorbar_title="Frac positive",
    title="Fraction of neuron activations explained by key freq",
    color=to_numpy(fraction_of_activations_positive_at_posn2),
)

뉴런이 활성화되는 데이터 포인트의 비율에 따라 색상을 지정합니다. 하나의 주파수로 잘 설명되는(frac > 0.85) 5개의 뚜렷한 뉴런 클러스터가 있음을 알 수 있습니다.

항상 활성화되는 여섯 번째의 분산된 뉴런 클러스터가 존재합니다. 이들은 특정 주파수에 의해 잘 설명되지 않습니다. 이는 ReLU가 이 클러스터에서 identity로 작동하기 때문에 타당합니다. 즉, 뉴런 basis를 특별히 선호할 이유가 없습니다 (다시 말해, 항상 활성화되는 뉴런들에 회전을 적용하는 것이 가능하므로, 이 뉴런 activation의 특정 값이 입력의 Fourier 성분과 관련하여 특별한 의미를 가질 것이라고 기대할 이유가 없습니다).

In [ ]:
# To represent that they are in a special sixth cluster, we set the frequency of these neurons to -1
neuron_freqs[neuron_frac_explained < 0.85] = -1.0
key_freqs_plus = t.concatenate([key_freqs, -key_freqs.new_ones((1,))])

for i, k in enumerate(key_freqs_plus):
    print(f"Cluster {i}: freq k={k}, {(neuron_freqs == k).sum()} neurons")

### 뉴런 클러스터의 추가 조사

각 클러스터에 대해 뉴런 activation의 Fourier Components의 norm을 개별적으로 확인할 수 있습니다. 다음 코드는 앞서 확인한 평균 $B_{i, j}^2$ 값의 plot과 동일한 작업을 수행하지만, 평균을 내기 전에 뉴런들을 주파수별로 클러스터로 정렬합니다.

*(참고로, 모든 plot을 한 번에 보기 위해 `animation_frame` 대신 `facet_col` 인자를 사용합니다.)*

In [ ]:
fourier_norms_in_each_cluster = []
for freq in key_freqs:
    fourier_norms_in_each_cluster.append(
        einops.reduce(
            neuron_acts_centered_fourier.pow(2)[..., neuron_freqs == freq],
            "batch_y batch_x neuron -> batch_y batch_x",
            "mean",
        )
    )

utils.imshow_fourier(
    t.stack(fourier_norms_in_each_cluster),
    title="Norm of 2D Fourier components of neuron activations in each cluster",
    facet_col=0,
    facet_labels=[f"Freq={freq}" for freq in key_freqs],
)

이제 뉴런 클러스터로 보이는 것들을 찾았으므로, 관찰 결과를 검증할 차례입니다. 각 뉴런 클러스터에 대해, 다른 모든 주파수 항을 0으로 설정하더라도 해당 태스크에서 여전히 좋은 성능을 낼 수 있음을 보여줌으로써 이를 검증하겠습니다.

### 연습 문제 - neuron cluster 검증하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 25-40 minutes on the following exercises.
> They are designed to get you to engage with ideas from linear algebra (specifically projections), and are very important conceptually.
> ```

우리는 다음과 같은 작업을 수행하고자 합니다:

* `neuron_acts_post`을 가져옵니다. 이는 `(p*p, d_mlp)` 형태의 tensor이며, `[i, j]`번째 요소는 `all_data` batch의 `i`번째 입력 sequence에 대한 neuron `j`의 activation입니다.
* 이 tensor를 $\mathbb{R}^{p^2}$ 내의 `p`개의 개별 벡터로 취급하여(마지막 차원이 batch 차원), 각 벡터를 해당 neuron의 관련 주파수에 대한 2D Fourier basis 벡터(즉, 상수항, 일차항, 이차항)로 생성된 $\mathbb{R}^{p^2}$의 subspace로 project 합니다.
* 이렇게 project 된 `neuron_acts_post` 벡터들을 가져와 `W_logit`을 적용하여 새로운 logit을 생성합니다. 이 logit을 사용한 cross entropy loss를 원래의 logit과 비교합니다.

만약 우리의 가설(즉, 특정 주파수와 관련된 각 neuron cluster에 대해 해당 주파수만이 중요하다는 가설)이 맞다면, 이런 방식으로 다른 주파수들을 project out 했을 때 loss가 크게 감소하지 않아야 합니다.

먼저, `project_onto_direction` 함수를 작성해야 합니다. 이 함수는 `batch_vecs`(마지막에 batch 차원이 있는 벡터들의 batch)과 `v`(단일 벡터)라는 두 개의 입력을 받으며, `batch_vecs`의 각 벡터를 방향 `v`으로 project 한 결과를 반환합니다.

In [ ]:
def project_onto_direction(batch_vecs: Tensor, v: Tensor) -> Tensor:
    """
    Returns the component of each vector in `batch_vecs` in the direction of `v`.

    batch_vecs.shape = (n, ...)
    v.shape = (n,)
    """
    raise NotImplementedError()


tests.test_project_onto_direction(project_onto_direction)

<details>
<summary>힌트</summary>

$v$(정규화된 벡터)에 대하여, $w$의 $v$ 위로의 projection은 다음과 같이 주어집니다:

$$
w_{proj} = (w \cdot v) v
$$

이 작업을 두 단계로 나누어 수행하는 것이 더 쉬울 수 있습니다: 먼저 0번째 차원을 따라 내적을 구하여 $v$ 방향의 성분을 계산한 다음, 이 성분들로 스케일링된 $v$의 배수 배치(batch)를 생성합니다.
</details>


<details><summary>솔루션</summary>

```python
def project_onto_direction(batch_vecs: Tensor, v: Tensor) -> Tensor:
    """
    Returns the component of each vector in `batch_vecs` in the direction of `v`.

    batch_vecs.shape = (n, ...)
    v.shape = (n,)
    """
    # Get tensor of components of each vector in v-direction
    components_in_v_dir = einops.einsum(batch_vecs, v, "n ..., n -> ...")

    # Use these components as coefficients of v in our projections
    return einops.einsum(components_in_v_dir, v, "..., n -> n ...")
```
</details>

다음으로, 함수 `project_onto_frequency`를 작성해야 합니다. 이 함수는 `(p**2, batch)` 형상을 가진 벡터 배치와 주파수 `freq`를 입력으로 받아, 해당 주파수에 대한 9개의 2D Fourier basis 벡터(즉, 하나의 상수항, 4개의 선형항, 4개의 이차항)로 생성된 subspace로의 각 벡터 투영(projection) 값을 반환합니다.

In [ ]:
def project_onto_frequency(batch_vecs: Tensor, freq: int) -> Tensor:
    """
    Returns the projection of each vector in `batch_vecs` onto the
    2D Fourier basis directions corresponding to frequency `freq`.

    batch_vecs.shape = (p**2, ...)
    """
    assert batch_vecs.shape[0] == p**2

    raise NotImplementedError()


tests.test_project_onto_frequency(project_onto_frequency)

<details>
<summary>힌트</summary>

기저 벡터들이 서로 직교하므로, 이는 위에서 작성한 `project_onto_direction` 함수를 9번 호출하여 합산하는 과정만으로 가능합니다 (투영하려는 각 기저 벡터에 대해 한 번씩 호출합니다).

투영할 벡터들을 가져오기 위해 `fourier_2d_basis_term` 함수를 사용해야 합니다. 크기가 `(p, p)`인 벡터가 아니라 길이가 `p**2`인 벡터를 다루고 있으므로, 이 벡터들을 flatten 하는 것을 잊지 마십시오!
</details>

<details>
<summary>솔루션</summary>

이 코드의 for 루프는 상수항 `(0, 0)`, 일차항 `(2f-1, 0)`, `(2f, 0)`, `(0, 2f-1)`, `(0, 2f)`, 그리고 이차항 `(2f-1, 2f-1)`, `(2f-1, 2f)`, `(2f, 2f-1)`, `(2f, 2f)`의 인덱스를 순회합니다.

```python
def project_onto_frequency(batch_vecs: Tensor, freq: int) -> Tensor:
    """
    Returns the projection of each vector in `batch_vecs` onto the
    2D Fourier basis directions corresponding to frequency `freq`.

    batch_vecs.shape = (p**2, ...)
    """
    assert batch_vecs.shape[0] == p**2

    return sum(
        [
            project_onto_direction(
                batch_vecs,
                fourier_2d_basis_term(i, j).flatten(),
            )
            for i in [0, 2 * freq - 1, 2 * freq]
            for j in [0, 2 * freq - 1, 2 * freq]
        ]
    )
```

</details>

마지막으로, 다음 코드를 실행하여 neuron activation에서 다른 주파수 성분들을 투영(project out)해 제거하고, 새로운 loss를 비교해 보십시오. 이 코드가 어떤 동작을 수행하는지 반드시 이해하시기 바랍니다.

In [ ]:
logits_in_freqs = []

for freq in key_freqs:
    # Get all neuron activations corresponding to this frequency
    filtered_neuron_acts = neuron_acts_post[:, neuron_freqs == freq]

    # Project onto const/linear/quadratic terms in 2D Fourier basis
    filtered_neuron_acts_in_freq = project_onto_frequency(filtered_neuron_acts, freq)

    # Calcluate new logits, from these filtered neuron activations
    logits_in_freq = filtered_neuron_acts_in_freq @ W_logit[neuron_freqs == freq]

    logits_in_freqs.append(logits_in_freq)

# We add on neurons in the always firing cluster, unfiltered
logits_always_firing = neuron_acts_post[:, neuron_freqs == -1] @ W_logit[neuron_freqs == -1]
logits_in_freqs.append(logits_always_firing)

# Compute new losses
key_freq_loss = utils.test_logits(sum(logits_in_freqs), bias_correction=True, original_logits=original_logits)
key_freq_loss_no_always_firing = utils.test_logits(
    sum(logits_in_freqs[:-1]), bias_correction=True, original_logits=original_logits
)

# Print new losses
print(f"""Loss with neuron activations ONLY in key freq (including always firing cluster): {key_freq_loss:.3e}
Loss with neuron activations ONLY in key freq (exclusing always firing cluster): {key_freq_loss_no_always_firing:.3e}
Original loss: {original_loss:.3e}
""")

다른 주파수들을 project out 했을 때 loss가 크게 변하지 않는다는 것을 알 수 있으며, 항상 활성화되는(always firing) cluster를 제거하더라도 loss는 여전히 매우 작습니다.

각 뉴런 클러스터를 ablation함으로써 (각 클러스터를 해당 주파수로 계속 제한하면서) 각 클러스터의 중요도를 비교할 수 있습니다. 이를 통해 `freq=52`이 가장 중요한 클러스터임을 알 수 있습니다 (이 클러스터가 제거되었을 때 loss가 크게 증가하기 때문입니다). 하지만 모든 클러스터가 중요함이 분명합니다 (어느 하나를 ablation하더라도 loss가 여전히 매우 작기 때문입니다). 또한 항상 활성화되는(always firing) 클러스터를 ablation하는 것은 영향이 매우 적으며, 따라서 이 클러스터가 해당 태스크에 그리 도움이 되지 않는다는 것을 알 수 있습니다. (이는 ReLU가 전혀 작동하지 않으면 본질적으로 선형 함수가 되며, 일반적으로 non-linearity가 있어야 훨씬 더 표현력이 풍부한 함수를 학습할 수 있다는 점에서 미리 짐작할 수 있었던 부분입니다.)

In [ ]:
print("Loss with neuron activations excluding none:     {:.9f}".format(original_loss.item()))
for c, freq in enumerate(key_freqs_plus):
    print(
        "Loss with neuron activations excluding freq={}:  {:.9f}".format(
            freq,
            utils.test_logits(
                sum(logits_in_freqs) - logits_in_freqs[c],
                bias_correction=True,
                original_logits=original_logits,
            ),
        )
    )

## Logit 계산 이해하기

> 요약: 네트워크는 $W_{logit}=W_{out}W_U$을 사용하여 $\cos(w(x+y)),\sin(w(x+y))$에 해당하는 방향 이외의 모든 2D Fourier 성분을 제거하고, 이후 이 방향들에 각각 $\cos(wz),\sin(wz)$를 곱하고 합산하여 최종 output logit을 얻습니다.

(관련 빈도 $k$ 가 있는 각 neuron cluster에 대해), 각 neuron의 activation은 상수항, 선형항 및 이차항의 선형 결합임을 상기하십시오:

$$
\begin{bmatrix}
1 & \cos(\omega_k x) & \sin(\omega_k x) \\
\cos(\omega_k y) & \cos(\omega_k x)\cos(\omega_k y) & \sin(\omega_k x)\cos(\omega_k y) \\
\sin(\omega_k y) & \cos(\omega_k x)\sin(\omega_k y) & \sin(\omega_k x)\sin(\omega_k y)
\end{bmatrix}
$$

이는 $\omega_k = 2\pi k / p$ 에 해당합니다.

logit을 계산하기 위해, 네트워크는 다음을 제외한 모든 방향을 상쇄합니다:

$$
\begin{aligned}
\cos(\omega_k (x+y)) &= \cos(\omega_k x)\cos(\omega_k y)-\sin(\omega_k x)\sin(\omega_k y) \\
\sin(\omega_k (x+y)) &= \sin(\omega_k x)\cos(\omega_k y)+\cos(\omega_k x)\sin(\omega_k y)
\end{aligned}
$$

그 다음 네트워크는 여기에 $\cos(wz),\sin(wz)$ 를 곱하고 합산합니다 (즉, 값 $z$ 에 대한 logit은 이 값들과 $\cos(wz),\sin(wz)$ 의 곱이 되며, 해당 cluster 내의 모든 neuron에 대해 합산되고, 다시 모든 cluster에 대해 합산됩니다).

#### 질문 - 이 알고리즘이 왜 작동하는지 설명해 주실 수 있나요?

<details>
<summary>힌트</summary>

다음 식에 대해 생각해 보십시오:

$$
\cos(\omega (x+y))\cos(\omega z)+\sin(\omega (x+y))\sin(\omega z)
$$

이는 (스케일 인자를 제외하면) $z$에 대한 logit 점수에 더해지는 값입니다.
</details>

<details>
<summary>답변</summary>

그 이유는 다시 한번 삼각함수 공식 덕분입니다! 각 neuron은 최종 logits에 다음과 같은 형태의 벡터를 더하게 됩니다:

$$
\cos(w(x+y))\cos(wz)+\sin(w(x+y))\sin(wz)
$$

우리는 삼각함수 공식을 통해 이것이 다음과 같음을 알고 있습니다:

$$
\cos(w(x+y-z))
$$

이는 $z=x+y$일 때 가장 큽니다.

---

이를 다르게 표현하면, 입력 `(x, y)`에 대해 모델의 logit 출력은 (스케일 인자를 제외하면) 다음과 같습니다:

$$
\cos(\omega(x+y-\vec{\textbf{z}})) = \begin{bmatrix}
\cos(\omega(x+y)) \\
\cos(\omega(x+y-1)) \\
\vdots \\
\cos(\omega(x+y-(p-1)))
\end{bmatrix}
$$

이 벡터는 인덱스 $x+y$인 요소에서 가장 크며, 이는 $x+y$에 대한 logit이 가장 커짐을 의미합니다 (이것이 바로 우리가 문제를 해결하기 위해 원하는 결과입니다!).

또한, 우리는 여러 다른 주파수 $\omega_k$를 가지고 있다는 점을 기억하십시오. 따라서 neuron들에 대해 합산할 때, 벡터들은 $z = x+y$에서 보강 간섭을 일으키고 그 외의 모든 곳에서는 상쇄 간섭을 일으킵니다:

$$
f(t) = \sum_{k \in K} C_k \cos(\omega_k (x + y - \vec{\textbf{z}}))
$$

(여기서 $C_k$는 큰 양의 상수이며, 나중에 명시적으로 계산할 것입니다).
</details>

### Fourier Basis에서의 Logits

네트워크가 다른 방향들을 상쇄시킨다는 것을 확인하기 위해, neuron activation과 logits를 모두 2D Fourier Basis로 변환하고 각 Fourier component에 해당하는 벡터의 norm을 보여줄 수 있습니다. **우리는 quadratic term이 neuron activation보다 logits에서 *훨씬* 더 높은 norm을 가지며, linear term은 0에 가깝다는 것을 알 수 있습니다.** activation에서 logits로 가기 위해서는 모든 neuron의 출력을 합산하는 선형 맵 $W_{logit}=W_{out}W_U$을 적용한다는 점을 기억하십시오.

아래는 이를 시각화하기 위한 코드입니다. 아래 코드에서 `mean` 메서드 대신 `einops.reduce`을 사용한 점에 유의하십시오. einops의 다른 대부분의 사용 사례와 마찬가지로, 이는 명시적이고 가독성이 좋기 때문에 유용합니다.

In [ ]:
utils.imshow_fourier(
    einops.reduce(neuron_acts_centered_fourier.pow(2), "y x neuron -> y x", "mean"),
    title="Norm of Fourier Components of Neuron Acts",
)

# Rearrange logits, so the first two dims represent (x, y) in modular arithmetic equation
original_logits_sq = einops.rearrange(original_logits, "(x y) z -> x y z", x=p)
original_logits_fourier = fft2d(original_logits_sq)
utils.imshow_fourier(
    einops.reduce(original_logits_fourier.pow(2), "y x z -> y x", "mean"),
    title="Norm of Fourier Components of Logits",
)

linear 항과 constant 항이 quadratic 항에 비해 거의 사라졌으며, quadratic 항이 neuron activation보다 logit에서 훨씬 더 크다는 것을 확인하실 수 있습니다. 이는 아래의 그래프에 표시되어 있습니다 (코드 실행 결과와 일치해야 합니다):

### 연습 문제 - quadratic 항만 사용하여 검증하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> This exercise should feel similar to the previous one, since it's about vectors and projections.
> ```

여기서는 각 주요 주파수 $k$에 대해 $\cos(\omega_k(x+y))$ 및 $\sin(\omega_k(x+y))$에 해당하는 logit 성분*만* 취함으로써 결과를 더욱 검증하고, 이것이 성능을 향상시킴을 보여줄 것입니다.

먼저, 주파수 `k`를 입력받아 다음 방향들에 해당하는 2D Fourier 공간의 `(p, p)` 크기 벡터들을 반환하는 함수 `get_trig_sum_directions`를 작성해야 합니다:

$$
\begin{aligned}
\cos(\omega_k (\vec{\textbf{x}}+\vec{\textbf{y}})) &= \cos(\omega_k \vec{\textbf{x}})\cos(\omega_k \vec{\textbf{y}})-\sin(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}}) \\
\sin(\omega_k (\vec{\textbf{x}}+\vec{\textbf{y}})) &= \sin(\omega_k \vec{\textbf{x}})\cos(\omega_k \vec{\textbf{y}})+\cos(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}})
\end{aligned}
$$

반환하는 벡터들은 반드시 정규화(normalized)되어야 함을 기억하십시오.

In [ ]:
def get_trig_sum_directions(k: int) -> tuple[Float[Tensor, "p p"], Float[Tensor, "p p"]]:
    """
    Given frequency k, returns the normalized vectors in the 2D Fourier basis representing the
    two directions:

        cos(ω_k * (x + y))
        sin(ω_k * (x + y))

    respectively.
    """
    raise NotImplementedError()


tests.test_get_trig_sum_directions(get_trig_sum_directions)

<details>
<summary>힌트</summary>

다음과 같이 벡터 $\cos(\omega_k \vec{\textbf{x}}) \cos(\omega_k \vec{\textbf{y}})$를 얻을 수 있습니다:

```python
cosx_cosy_direction = fourier_2d_basis_term(2*k-1, 2*k-1)
```
</details>


<details><summary>솔루션</summary>

```python
def get_trig_sum_directions(k: int) -> tuple[Float[Tensor, "p p"], Float[Tensor, "p p"]]:
    """
    Given frequency k, returns the normalized vectors in the 2D Fourier basis representing the
    two directions:

        cos(ω_k * (x + y))
        sin(ω_k * (x + y))

    respectively.
    """
    cosx_cosy_direction = fourier_2d_basis_term(2 * k - 1, 2 * k - 1)
    sinx_siny_direction = fourier_2d_basis_term(2 * k, 2 * k)
    sinx_cosy_direction = fourier_2d_basis_term(2 * k, 2 * k - 1)
    cosx_siny_direction = fourier_2d_basis_term(2 * k - 1, 2 * k)

    cos_xplusy_direction = (cosx_cosy_direction - sinx_siny_direction) / np.sqrt(2)
    sin_xplusy_direction = (sinx_cosy_direction + cosx_siny_direction) / np.sqrt(2)

    return cos_xplusy_direction, sin_xplusy_direction
```
</details>

이 테스트들을 통과했다면, 코드를 실행하여 logit을 이 방향들로 투영하고 loss가 어떻게 변하는지 확인할 수 있습니다. 앞서 작성하신 `project_onto_direction` 함수의 사용에 유의하시기 바랍니다.

In [ ]:
trig_logits = []

for k in key_freqs:
    cos_xplusy_direction, sin_xplusy_direction = get_trig_sum_directions(k)
    cos_xplusy_projection = project_onto_direction(original_logits, cos_xplusy_direction.flatten())
    sin_xplusy_projection = project_onto_direction(original_logits, sin_xplusy_direction.flatten())

    trig_logits.extend([cos_xplusy_projection, sin_xplusy_projection])

trig_logits = sum(trig_logits)

print(f"Loss with just x+y components: {utils.test_logits(trig_logits, True, original_logits):.4e}")
print(f"Original Loss: {original_loss:.4e}")

이 구성 요소들만 사용했을 때의 loss가 원래의 loss보다 상당히 **낮은** 것을 확인하실 수 있을 것입니다. 이는 우리가 모델이 사용하는 알고리즘을 정확하게 식별했다는 매우 강력한 증거입니다.

### Fourier Basis에서의 $W_{logit}$

좋습니다, 이제 우리는 모델이 각 주요 주파수 $k$에 대해 주로 $\cos(\omega_k(x+y))$ 및 $\sin(\omega_k(x+y))$ 항을 사용하여 작동한다는 것을 알고 있습니다. 이제 모델의 최종 출력이 다음과 같음을 보여주고자 합니다:

$$
\cos(\omega_k (x + y - \vec{\textbf{z}})) = \cos(\omega_k (x + y))\cos(\omega_k \vec{\textbf{z}}) + \sin(\omega_k (x + y))\sin(\omega_k \vec{\textbf{z}})
$$

이를 어떻게 수행할까요?

정답: ***Fourier basis에서 $W_{logit}$를 조사합니다.*** 만약 $W_{logit}$이 주로 $\cos(\omega_k \vec{\textbf{z}})$ 및 $\sin(\omega_k \vec{\textbf{z}})$ 방향으로 projection하고 있다고 생각한다면, 다음과 같은 결과를 기대할 수 있습니다:

$$
W_{logit} \approx U S F = \sum_{i=1}^p \sigma_i u_i f_i^T
$$

여기서 singular values $\sigma_i$은 주요 주파수 $k$에 해당하는 Fourier basis 벡터 $f_i = \cos(\omega_k \vec{\textbf{z}}), \sin(\omega_k \vec{\textbf{z}})$을 제외하고는 모두 0입니다. 다시 말해:

$$
W_{logit} \approx \sum_{k \in K} \sigma_{2k-1} u_{2k-1} \cos(\omega_k \vec{\textbf{z}}) + \sigma_{2k} u_{2k} \sin(\omega_k \vec{\textbf{z}})
$$

따라서 $W_{logit}$에 $F^T$를 우측 곱셈(right-multiply)하면, shape이 `(d_mlp, p)`인 행렬 $W_{logit} F^T \approx US$을 얻게 되며, 이 행렬은 주요 주파수 $k$에 대한 $\sigma_{2k-1}u_{2k-1}$ 및 $\sigma_{2k} u_{2k}$를 제외한 모든 열이 0이 됩니다. 이를 확인해 보겠습니다:

In [ ]:
US = W_logit @ fourier_basis.T

utils.imshow_div(US, x=fourier_basis_names, yaxis="Neuron index", title="W_logit in the Fourier Basis")

이 행렬의 열들은 key frequency에 해당하는 위치 $2k$, $2k-1$에서만 0이 아님을 확인할 수 있습니다. 우리 모델의 최종 출력은 단순히 이러한 열들의 선형 결합(neuron activation에 의해 결정되는 계수 사용)이므로, 이는 $W_{logit}$가 key frequency에 해당하는 방향으로 projection하고 있음을 증명합니다.

> 우리가 방금 수행한 작업과 이전에 했던 작업의 차이점에 주목하십시오. 이전 섹션들에서는 입력 공간에 대해 activation / effective weight의 2D Fourier transform을 수행했습니다 (벡터는 $\vec{\textbf{x}}$ 및 $\vec{\textbf{y}}$였습니다). 지금은 출력 공간에 대해 1D Fourier transform을 수행하고 있습니다 (벡터는 $\vec{\textbf{z}})$입니다). 이것이 작동한다는 점이 매우 흥미롭습니다!

따라서 우리는 다음을 증명했습니다:

$$
W_{logit} \approx \sum_{k \in K} \sigma_{2k-1} u_{2k-1} \cos(\omega_k \vec{\textbf{z}}) + \sigma_{2k} u_{2k} \sin(\omega_k \vec{\textbf{z}})
$$

하지만 우리는 여전히 최종 출력이 다음과 같음을 보여주고 싶습니다:

$$
f(t) = n_{post}^T W_{logit} \approx \sum_{k \in K} C_k \big(\cos(\omega_k (x + y)) \cos(\omega_k \vec{\textbf{z}}) + \sin(\omega_k (x + y)) \sin(\omega_k \vec{\textbf{z}})\big)
$$

여기서 $n_{post} \in \mathbb{R}^{d_{mlp}}$는 neuron activation 벡터이며, $C_k$은 큰 양의 상수입니다.

벡터 $\cos(\omega_k \vec{\textbf{z}})$와 $\sin(\omega_k \vec{\textbf{z}})$의 계수를 맞추면, 이는 우리가 다음을 보여주고 싶다는 것을 의미합니다:

$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx C_k \cos(\omega_k (x + y)) \\
\sigma_{2k} u_{2k} &\approx C_k \sin(\omega_k (x + y))
\end{aligned}
$$

각 key frequency $k$에 대하여 말입니다.

먼저, 간단한 sanity check를 수행해 보겠습니다. 우리는 벡터 $u_{2k-1}$과 $u_{2k}$가 frequency $k$의 성분만을 포함할 것으로 예상하며, 이는 이 벡터들의 0이 아닌 유일한 요소들이 $k$-frequency cluster의 neuron들에 대응될 것임을 의미합니다. 각 cluster의 neuron들이 함께 그룹화되도록 행렬 $W_{logit}F^T \approx US$를 재배열하여 이를 테스트해 보겠습니다:

In [ ]:
US_sorted = t.concatenate([US[neuron_freqs == freq] for freq in key_freqs_plus])
hline_positions = np.cumsum([(neuron_freqs == freq).sum().item() for freq in key_freqs]).tolist() + [cfg.d_mlp]

utils.imshow_div(
    US_sorted,
    x=fourier_basis_names,
    yaxis="Neuron",
    title="W_logit in the Fourier Basis (rearranged by neuron cluster)",
    hline_positions=hline_positions,
    hline_labels=[f"Cluster: {freq=}" for freq in key_freqs.tolist()] + ["No freq"],
)

각 주파수 $k$에 대해, 방향 $\cos(\omega_k \vec{\textbf{z}})$ 및 $\sin(\omega_k \vec{\textbf{z}})$의 출력 성분은 $k$-cluster의 neuron들에 의해서만 결정된다는 것을 알 수 있습니다. 즉, 이들은 주파수 $k$를 가진 입력 $(x, y)$의 2D Fourier 성분들에 의해서만 결정됩니다.

이는 유망한 결과이지만, 우리는 아직 $\sigma_{2k-1} u_{2k-1} \propto \cos(\omega_k(x+y))$ 등을 증명하지 못했습니다. 이를 위해, 모든 입력 $(x, y)$에 대해 벡터 $\sigma_{2k-1} u_{2k-1}$ 및 $\sigma_{2k} u_{2k}$을 계산한 다음, 2D Fourier transform을 수행하겠습니다.

In [ ]:
cos_components = []
sin_components = []

for k in key_freqs:
    sigma_u_sin = US[:, 2 * k]
    sigma_u_cos = US[:, 2 * k - 1]

    logits_in_cos_dir = neuron_acts_post_sq @ sigma_u_cos
    logits_in_sin_dir = neuron_acts_post_sq @ sigma_u_sin

    cos_components.append(fft2d(logits_in_cos_dir))
    sin_components.append(fft2d(logits_in_sin_dir))

for title, components in zip(["Cosine", "Sine"], [cos_components, sin_components]):
    utils.imshow_fourier(
        t.stack(components),
        title=f"{title} components of neuron activations in Fourier basis",
        animation_frame=0,
        animation_name="Frequency",
        animation_labels=key_freqs.tolist(),
    )

이 플롯을 해석할 수 있습니까? 이 플롯이 logit이 계산되는 방식에 대한 우리의 가설을 어떻게 확인해 주는지 설명할 수 있습니까?

<details>
<summary>출력 (및 설명)</summary>

우리가 증명하려는 내용은 다음과 같습니다:

$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx C_k \cos(\omega_k (x + y)) \\
\sigma_{2k} u_{2k} &\approx C_k \sin(\omega_k (x + y))
\end{aligned}
$$

이를 2D Fourier basis로 작성하면 다음과 같습니다:

$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx \frac{C_k}{\sqrt{2}} \cos(\omega_k \vec{\textbf{x}})\cos(\omega_k \vec{\textbf{y}}) - \frac{C_k}{\sqrt{2}} \sin (\omega_k \vec{\textbf{x}})\sin (\omega_k \vec{\textbf{y}}) \\
\sigma_{2k} u_{2k} &\approx \frac{C_k}{\sqrt{2}} \cos(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}}) + \frac{C_k}{\sqrt{2}} \sin (\omega_k \vec{\textbf{x}})\cos (\omega_k \vec{\textbf{y}})
\end{aligned}
$$

이 예상 2D Fourier 계수들이 플롯에서 얻은 계수들과 일치한다는 것을 알 수 있을 것입니다 (즉, 크기가 거의 같고, $\sin$의 경우 부호가 같으며 $\cos$의 경우 부호가 반대입니다). 예를 들어, 주파수 $k=14$에 대한 $\cos$ 플롯을 확대해 보면 다음과 같습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/cosine-components.png" width="650">
</details>

## 섹션 요약

네트워크의 각 부분에 대해 학습한 내용을 복습해 보겠습니다. 우리는 다음과 같은 사실을 발견했습니다:

> #### Embedding
>
> embedding은 소수의 **key frequencies**에 해당하는 선택된 몇 개의 1D Fourier basis 벡터로 투영됩니다.
>
> #### Neuron Activations
>
> 각 neuron의 activation은 상수항과 **특정 주파수의 선형 및 이차항**의 선형 결합입니다. neuron들은 **key frequencies에 따라 명확하게 클러스터링**됩니다.
>
> 우리는 다음 방법을 통해 이를 발견했습니다:
> * 특정 주파수가 neuron의 분산을 얼마나 잘 설명하는지 확인하였으며, 모든 neuron이 단일 주파수에 의해 잘 설명된다는 것을 발견했습니다 (항상 활성화되는, 즉 비선형 작업을 수행하지 않는 일부 neuron 그룹 제외).
> * neuron activation을 해당 주파수들에 투영하였고, 이것이 전체 loss를 크게 증가시키지 않음을 보여주었습니다.
>
> 이러한 activation은 ReLU와 attention layer가 포함된 어떤 비선형적인 마법에 의해 계산됩니다 (왜냐하면 서로 다른 두 시퀀스 위치의 입력 곱과 같은 것을 계산하는 것은 transformer가 학습하기에 자연스러운 연산이 아니기 때문입니다!).
>
> #### Logit Computation
>
> 네트워크는 $W_{logit}=W_{out}W_U$를 사용하여 $\cos(w(x+y)),\sin(w(x+y))$에 해당하는 방향을 제외한 모든 2D Fourier 성분을 상쇄시키고, 이후 이 방향들에 각각 $\cos(wz),\sin(wz)$를 곱하고 합산하여 출력 logit을 얻습니다.
>
> 우리는 다음 방법을 통해 이를 발견했습니다:
>
> * 선형 항들이 logit에서 거의 사라지고, 이차항들만 남는다는 것을 보여주었습니다.
> * 각 neuron은 특정 주파수와 연관되어 있는 것으로 보였으며, 주파수 클러스터 $k$에 속한 neuron의 출력은 $k$번째 Fourier basis의 logit에 직접적인 영향을 주었습니다.
> * neuron이 이 주파수의 logit에 영향을 주는 정확한 방식은 우리의 초기 추측인 $\cos(\omega_k (x + y - \vec{\textbf{z}})) = \cos(\omega_k (x + y))\cos(\omega_k \vec{\textbf{z}}) + \sin(\omega_k (x + y))\sin(\omega_k \vec{\textbf{z}})$와 일치했습니다. 즉, 각 주파수 클러스터는 이 선형 결합을 계산하여 logit 출력에 더하는 역할을 담당했습니다.

# 3️⃣ 학습 중 분석

> ##### 학습 목표
>
> * 시간에 따른 metric을 추적하는 개념과, 이를 통해 특정 circuit이 언제 형성되는지 파악하는 방법을 이해합니다.
> * 모델 weight 행렬의 singular value가 시간에 따라 어떻게 진화하는지 조사하고 해석합니다.
> * 교환법칙(commutativity)과 같이 모델 내의 다른 능력들이 어떻게 형성되는지 조사합니다.

## 시작하며 참고할 사항

*참고 - 이 섹션은 이전 섹션들보다 연습 문제가 적으며, 논문의 결과물 중 일부를 보여주는 데 더 중점을 둡니다.*

이 섹션에서는 학습 과정 중의 modular addition transformer를 분석합니다. 우리가 사용할 데이터에서는 epoch 0부터 50K까지 매 100 epoch마다 checkpoint를 저장했습니다 (이전 연습 문제에서 사용한 모델은 40K 시점의 모델입니다).

사용성 참고: 이 섹션에서는 애니메이션을 자주 사용합니다. 재생 버튼을 누르기보다 슬라이더를 수동으로 조작하는 것을 권장합니다. Plotly가 애니메이션을 부드럽게 처리하는 방식이 혼란스럽고 오해의 소지가 있기 때문입니다 (아직 해결 방법을 찾지 못했습니다).

사용성 참고 2: 그래프를 표시하기 위해 데이터 포인트가 너무 많으면 안 되므로, 그래프마다 데이터 포인트 사이의 epoch 간격이 다르거나 최종 epoch가 다를 수 있습니다.

표기법: $\cos(\omega(x+y))$, $\sin(\omega(x+y))$ 컴포넌트를 지칭하기 위해 "trig components"라는 용어를 사용합니다.

## 개요

* 모델은 phase change가 일어나기 훨씬 전부터 일반화된 알고리즘을 잘 학습하기 시작하며, 이는 상당히 부드럽게 진행됩니다.
    * 이는 excluded loss 지표를 통해 확인할 수 있으며, 모델이 $\cos(w(x+y))$ 방향을 사용하여 "더 효율적으로 암기"하는 법을 배운다는 것을 나타내는 것으로 보입니다.
    * 이는 'grokking은 loss landscape에서의 무작위 보행(random walk)이며 결국 운 좋게 정답을 찾는 것'이라는 가설을 명확히 반증합니다.
* 또한 모델 내의 각 circuit과 그것이 어떻게 발전하는지를 더 정성적으로 조사합니다.
    * 모든 circuit이 pre-grokking 단계에서 어느 정도 발전하지만, 그 속도는 서로 다르며 일부는 다른 것들보다 더 뚜렷한 phase change를 보입니다.
    * embedding circuit, 'trig dimensions 계산' circuit, 그리고 commutativity의 발전을 조사합니다.
    * 또한 neuron activation의 발전과 그것이 cluster별로 어떻게 다른지 탐색합니다.
* '모델이 일반화 가능한 알고리즘을 학습하는 시점'과 '모델이 모든 암기된 노이즈를 제거하는 시점' 사이에는 작지만 눈에 띄는 시차가 존재합니다.
* 주요 grokking 외에도 몇 가지 더 작은 phase change의 징후가 있습니다.
    * 특히 grokking이 일어난 한참 후인 43K-44K 지점에서 phase change가 나타납니다 (이 부분에서 정확히 어떤 일이 일어나고 있는지는 아직 해석하지 않았습니다).

## 설정

먼저, 몇 가지 유용한 함수들을 정의하겠습니다. 특히, `get_metrics` 함수는 학습 기간 동안의 metric 딕셔너리를 채우도록 설계되었습니다. 인자 `metric_fn`은 모델을 입력받아 metric을 반환하는 함수입니다 (예를 들어, 테스트 세트에 대한 모델의 loss를 반환하기 위해 `metric_fn=test_loss`를 사용합니다).

In [ ]:
# Define a dictionary to store our metrics in
metric_cache = {}


def get_metrics(model: HookedTransformer, metric_cache, metric_fn, name, reset=False):
    """
    Define a metric (by metric_fn) and add it to the cache, with the name `name`.

    If `reset` is True, then the metric will be recomputed, even if it is already in the cache.
    """
    if reset or (name not in metric_cache) or (len(metric_cache[name]) == 0):
        metric_cache[name] = []
        for sd in tqdm(full_run_data["state_dicts"]):
            model = utils.load_in_state_dict(model, sd)
            out = metric_fn(model)
            if isinstance(out, Tensor):
                out = to_numpy(out)
            metric_cache[name].append(out)
        model = utils.load_in_state_dict(model, full_run_data["state_dicts"][400])
        metric_cache[name] = t.tensor(np.array(metric_cache[name]))


def test_loss(model):
    logits = model(all_data)[:, -1, :-1]
    return utils.test_logits(logits, False, mode="test")


def train_loss(model):
    logits = model(all_data)[:, -1, :-1]
    return utils.test_logits(logits, False, mode="train")


epochs = full_run_data["epochs"]
plot_metric = partial(utils.lines, x=epochs, xaxis="Epoch")

get_metrics(model, metric_cache, test_loss, "test_loss")
get_metrics(model, metric_cache, train_loss, "train_loss")

## Excluded Loss

주파수 $w$에 대한 **Excluded Loss**는 $\cos(w(x+y))$ 및 $sin(w(x+y))$에 해당하는 logit 성분들을 삭제했을 때의 training set 상의 loss입니다. 우리는 주요 주파수의 각 $w$에 대해 별도의 메트릭을 얻습니다.

**핵심 관찰:** excluded loss(특히 주파수 14의 경우)는 grokking이 일어나는 시점보다 *훨씬* 이전에 상승하기 시작합니다.

(참고: 이러한 성능 저하는 무작위 방향을 삭제했을 때 얻게 되는 결과보다 훨씬 더 큽니다.)

### 연습 문제 - excluded loss 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 20-30 minutes on this exercise.
> ```

excluded loss를 구현하기 위해 아래 함수를 완성해야 합니다. 이전 연습 문제에서 다룬 `get_trig_sum_directions` 및 `project_onto_direction` 함수를 사용해야 합니다. 가이드라인으로 함수의 처음 몇 줄을 제공해 드렸습니다.

참고 - loss를 계산할 때는 `test_logits` 함수를 사용하며, 인자는 `bias_correction=False, mode="train"` 로 설정해야 합니다.

In [ ]:
def excl_loss(model: HookedTransformer, key_freqs: list) -> list:
    """
    Returns the excluded loss (i.e. subtracting the components of logits corresponding to
    cos(w_k(x+y)) and sin(w_k(x+y)), for each frequency k in key_freqs.
    """
    excl_loss_list = []
    logits = model(all_data)[:, -1, :-1]
    raise NotImplementedError()


tests.test_excl_loss(excl_loss, model, key_freqs)

<details><summary>솔루션</summary>

```python
def excl_loss(model: HookedTransformer, key_freqs: list) -> list:
    """
    Returns the excluded loss (i.e. subtracting the components of logits corresponding to
    cos(w_k(x+y)) and sin(w_k(x+y)), for each frequency k in key_freqs.
    """
    excl_loss_list = []
    logits = model(all_data)[:, -1, :-1]
    for freq in key_freqs:
        cos_xplusy_direction, sin_xplusy_direction = get_trig_sum_directions(freq)

        logits_cos_xplusy = project_onto_direction(logits, cos_xplusy_direction.flatten())
        logits_sin_xplusy = project_onto_direction(logits, sin_xplusy_direction.flatten())
        logits_excl = logits - logits_cos_xplusy - logits_sin_xplusy

        loss = utils.test_logits(logits_excl, bias_correction=False, mode="train").item()
        excl_loss_list.append(loss)

    return excl_loss_list
```
</details>

이 함수를 완성한 후, 다음 코드를 실행하여 각 주요 주파수에 대한 excluded loss를 플롯할 수 있습니다 (기준점으로 training loss와 testing loss도 함께 표시됩니다). 이 플롯은 Neel Nanda의 LessWrong 포스트 [Key Claims](https://www.lesswrong.com/posts/N6WM6hs7RQMKDhYjB/a-mechanistic-interpretability-analysis-of-grokking#Key_Claims) 섹션 끝에 있는 플롯과 일치해야 합니다.

In [ ]:
get_metrics(model, metric_cache, partial(excl_loss, key_freqs=key_freqs), "excl_loss")

plot_metric(
    t.concat(
        [
            metric_cache["excl_loss"].T,
            metric_cache["train_loss"][None, :],
            metric_cache["test_loss"][None, :],
        ]
    ),
    labels=[f"excl {freq}" for freq in key_freqs] + ["train", "test"],
    title="Excluded Loss for each trig component",
    log_y=True,
    yaxis="Loss",
)

## embedding의 발전

### Fourier basis에서의 embedding

우리는 각 epoch에서 각 1D Fourier component의 embedding norm을 그래프로 그릴 수 있습니다. grokking 이전 단계에서 모델은 몇 개의 component에 우선순위를 두는 표현을 학습하고 있지만, 대부분의 component는 여전히 무시할 수 없는 값을 가지고 있습니다. 이는 아마도 이러한 방향들이 암기 과정에서 일부 역할을 수행하고 있기 때문일 것입니다. 그 후, grokking 기간 동안 다른 component들은 거의 0으로 설정됩니다. 모델은 더 이상 무언가를 암기하기 위해 다른 방향들이 필요하지 않게 되었으며, 일반적인 알고리즘을 학습한 것입니다.

(상기시켜 드리자면, 우리는 embedding의 SVD가 대략 $W_E \approx F^T S V^T$ 임을 발견했습니다. 여기서 $F$ 은 Fourier basis 벡터이고 $S$ 는 sparse 하므로, $F W_E \approx S V^T$ 또한 대부분의 행이 0인 sparse 한 상태입니다. 따라서 학습 중간 지점에서 $F W_E$ 의 행 norm을 계산할 때, 우리는 embedding의 입력 공간이 어떻게 선택된 소수의 주파수만을 포함하도록 학습하는지를 확인하게 됩니다.)

### 연습 문제 - `fourier_embed` 정의하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> This exercise shouldn't take more than ~10 minutes.
> ```

함수 `fourier_embed`를 작성하십시오. 이 함수는 모델의 embedding matrix의 Fourier transformation의 norm을 계산해야 합니다 (`=`에 해당하는 embedding vector는 제외합니다). 다시 말해, embedding matrix에 Fourier basis matrix를 왼쪽에서 곱한 다음, 각 embedding vector의 norm의 합을 계산해야 합니다.

In [ ]:
def fourier_embed(model: HookedTransformer):
    """
    Returns norm of Fourier transform of the model's embedding matrix.
    """
    raise NotImplementedError()


tests.test_fourier_embed(fourier_embed, model)

<details><summary>솔루션</summary>

```python
def fourier_embed(model: HookedTransformer):
    """
    Returns norm of Fourier transform of the model's embedding matrix.
    """
    W_E_fourier = fourier_basis.T @ model.W_E[:-1]
    return einops.reduce(W_E_fourier.pow(2), "vocab d_model -> vocab", "sum")
```
</details>

다음으로, 학습 과정 동안 embedding의 Fourier component의 norm이 어떻게 변화하는지 그래프로 그릴 수 있습니다:

In [ ]:
# Plot every 200 epochs so it's not overwhelming
get_metrics(model, metric_cache, fourier_embed, "fourier_embed")

utils.animate_lines(
    metric_cache["fourier_embed"][::2],
    snapshot_index=epochs[::2],
    snapshot="Epoch",
    hover=fourier_basis_names,
    animation_group="x",
    title="Norm of Fourier Components in the Embedding Over Training",
)

### 연습 문제 - $W_E$의 SVD 조사하기

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

우리는 $W_E \approx F^T S V^T$가 SVD의 좋은 근사치라고 논의했지만, 실제로 SVD를 계산하면 어떤 일이 발생할까요?

학습 중간 지점에서 $W_E$의 singular value decomposition을 통해 singular values를 반환하는 다음 함수를 완성해야 합니다. (bias 항에 해당하는 $W_E$의 마지막 행을 제거하는 것을 잊지 마십시오.) PyTorch에는 이를 위해 사용할 수 있는 SVD 함수(`torch.svd`)가 있습니다.

In [ ]:
def embed_SVD(model: HookedTransformer) -> Tensor:
    """
    Returns vector S, where W_E = U @ diag(S) @ V.T in singular value decomp.
    """
    raise NotImplementedError()


tests.test_embed_SVD(embed_SVD, model)

In [ ]:
get_metrics(model, metric_cache, embed_SVD, "embed_SVD")

utils.animate_lines(
    metric_cache["embed_SVD"],
    snapshot_index=epochs,
    snapshot="Epoch",
    title="Singular Values of the Embedding During Training",
    xaxis="Singular Number",
    yaxis="Singular Value",
)

<details><summary>솔루션</summary>

```python
def embed_SVD(model: HookedTransformer) -> Tensor:
    """
    Returns vector S, where W_E = U @ diag(S) @ V.T in singular value decomp.
    """
    U, S, V = t.svd(model.W_E[:, :-1])
    return S
```
</details>

이 그래프에서 어떤 일이 일어나고 있는지 해석해 주시겠습니까?

<details>
<summary>해석</summary>

처음에는 SVD 값이 기본적으로 무작위입니다. 학습이 진행됨에 따라 작은 singular value들은 0으로 수렴하는 경향이 있으며, 핵심 주파수에 해당하는 singular value들은 증가합니다. 결과적으로, 그래프는 핵심 주파수에 해당하는 값들을 제외한 모든 singular value가 0인 sparse matrix를 보여줍니다.
</details>

***참고 - 이 지점 이후로는 더 이상 연습 문제가 없습니다. 내용은 단순히 시각화 및 결과 해석/논의로 구성됩니다.***

## 삼각 함수 성분 계산의 발전 과정

이전 연습 문제들에서, 우리는 각 주요 주파수 $k$에 대해 $\cos(\omega_k(x+y))$ 및 $\sin(\omega_k(x+y))$에 해당하는 2D Fourier basis 방향으로 logit 또는 neuron activation을 투영했습니다. 우리는 이러한 방향들이 모델 성능의 거의 전부를 설명한다는 것을 발견했습니다.

여기서는 동일한 작업을 수행하되, 시간에 따라 분석하여 모델이 이러한 삼각 함수 성분을 계산하는 방법을 어떻게 학습하는지 살펴보겠습니다. 코드는 아래에 모두 제공되어 있습니다 (이전 섹션에서 작성한 일부 함수들을 사용합니다).

activation은 먼저 중심화(centered)된 후, 제곱합이 계산됩니다. 그 다음 Fourier 성분들이 추출되며, 이들이 분산의 어느 정도 비율을 설명하는지 확인합니다. 이 값은 이후 "output dimension"(neuron activation의 경우 neuron, logit의 경우 output class)에 대해 평균을 냅니다.

In [ ]:
def tensor_trig_ratio(model: HookedTransformer, mode: str):
    """
    Returns the fraction of variance of the (centered) activations which is explained by the Fourier
    directions corresponding to cos(ω(x+y)) and sin(ω(x+y)) for all the key frequencies.
    """
    logits, cache = model.run_with_cache(all_data)
    logits = logits[:, -1, :-1]
    if mode == "neuron_pre":
        tensor = cache["pre", 0][:, -1]
    elif mode == "neuron_post":
        tensor = cache["post", 0][:, -1]
    elif mode == "logit":
        tensor = logits
    else:
        raise ValueError(f"{mode} is not a valid mode")

    tensor_centered = tensor - einops.reduce(tensor, "xy index -> 1 index", "mean")
    tensor_var = einops.reduce(tensor_centered.pow(2), "xy index -> index", "sum")
    tensor_trig_vars = []

    for freq in key_freqs:
        cos_xplusy_direction, sin_xplusy_direction = get_trig_sum_directions(freq)
        cos_xplusy_projection_var = (
            project_onto_direction(tensor_centered, cos_xplusy_direction.flatten()).pow(2).sum(0)
        )
        sin_xplusy_projection_var = (
            project_onto_direction(tensor_centered, sin_xplusy_direction.flatten()).pow(2).sum(0)
        )
        tensor_trig_vars.extend([cos_xplusy_projection_var, sin_xplusy_projection_var])

    return to_numpy(sum(tensor_trig_vars) / tensor_var)


for mode in ["neuron_pre", "neuron_post", "logit"]:
    get_metrics(
        model,
        metric_cache,
        partial(tensor_trig_ratio, mode=mode),
        f"{mode}_trig_ratio",
        reset=True,
    )

lines_list = []
line_labels = []
for mode in ["neuron_pre", "neuron_post", "logit"]:
    tensor = metric_cache[f"{mode}_trig_ratio"]
    lines_list.append(einops.reduce(tensor, "epoch index -> epoch", "mean"))
    line_labels.append(f"{mode}_trig_frac")

plot_metric(
    lines_list,
    labels=line_labels,
    log_y=False,
    yaxis="Ratio",
    title="Fraction of logits and neurons explained by trig terms",
)

로그 스케일로 플롯하면, 3가지 모두 학습 과정에서 trig 성분의 비율이 높아지는 것을 더 명확하게 볼 수 있습니다. 하지만 logit은 더 부드럽게 변화하는 반면, neuron은 더 뚜렷한 위상 변화(phase change)를 보입니다.

### 토론

완전히 학습된 모델에서, 모델이 logit에서 삼각함수 방향을 의미 있게 사용할 수 있게 만드는 알고리즘에는 두 가지 핵심 구성 요소가 있습니다. 첫째는 neuron activation에 상당한 이차항(quadratic terms)이 있다는 점이고, 둘째는 $W_{logit}$이 모든 비삼각함수 항을 상쇄시킨 후 삼각함수 항을 `(x + y) % p`로 매핑할 수 있다는 점입니다.

자연스러운 의문은 이 중 하나가 먼저 나타나는지, 아니면 둘 다 동시에 진화하는지 여부입니다. 제가 알기로는 "여러 개의 가동 부품을 가진 circuit이 학습 과정에서 어떻게 형성되는가"에 대해서는 전혀 알려진 바가 없습니다.

이 사례의 경우, logit은 초기에 삼각함수 방향을 제외한 모든 것을 상쇄하는 능력을 개발하며, neuron은 grokking 지점에 가까워질 때까지 상당한 이차항이나 삼각함수 성분을 개발하지 않습니다.

저는 circuit이 "역순으로" 개발되는 것이 더 타당하다는 막연한 추측을 하고 있습니다. 만약 좋은 출력을 생성하기 위해 두 개의 layer가 함께 작동해야 한다면, 두 번째 layer가 무작위로 초기화되었을 때 첫 번째 layer는 아무것도 할 수 없습니다. 하지만 첫 번째 layer가 무작위로 초기화되었다면, 두 번째 layer는 "정확한" 출력에 해당하는 출력 성분만을 추출하여 이를 통해 출력 솔루션을 부정확하게나마 근사하는 법을 배울 수 있습니다. 그리고 *이제* 네트워크는 circuit의 두 부분을 모두 구축해야 할 학습 동기를 갖게 됩니다.

(이는 아마도 Lottery Ticket 가설과 관련이 있을지도 모릅니다.)

## neuron activation의 발전

neuron activation에는 주목할 만한 두 가지 특징이 있습니다:
* x와 y가 동일한 주파수를 가진 quadratic terms의 상당한 성분을 포함하고 있습니다.
* 단일 주파수의 Fourier terms를 가진 클러스터로 그룹화됩니다.

우리는 해당 neuron 주파수의 quadratic terms(epoch 40K 모델에서 가져온 주파수)에 의해 설명되는 neuron의 (centered) activation 비율을 그래프로 그려 첫 번째 특징을 연구할 수 있습니다.

(항상 활성화되는 클러스터의 경우 모든 주파수에 대해 합산합니다).

In [ ]:
def get_frac_explained(model: HookedTransformer) -> Tensor:
    _, cache = model.run_with_cache(all_data, return_type=None)

    returns = []

    for neuron_type in ["pre", "post"]:
        neuron_acts = cache[neuron_type, 0][:, -1].clone().detach()
        neuron_acts_centered = neuron_acts - neuron_acts.mean(0)
        neuron_acts_fourier = fft2d(einops.rearrange(neuron_acts_centered, "(x y) neuron -> x y neuron", x=p))

        # Calculate the sum of squares over all inputs, for each neuron
        square_of_all_terms = einops.reduce(neuron_acts_fourier.pow(2), "x y neuron -> neuron", "sum")

        frac_explained = t.zeros(utils.d_mlp).to(device)
        frac_explained_quadratic_terms = t.zeros(utils.d_mlp).to(device)

        for freq in key_freqs_plus:
            # Get Fourier activations for neurons in this frequency cluster
            # We arrange by frequency (i.e. each freq has a 3x3 grid with const, linear & quadratic
            # terms)
            acts_fourier = arrange_by_2d_freqs(neuron_acts_fourier[..., neuron_freqs == freq])

            # Calculate the sum of squares over all inputs, after filtering for just this frequency
            # Also calculate the sum of squares for just the quadratic terms in this frequency
            if freq == -1:
                squares_for_this_freq = squares_for_this_freq_quadratic_terms = einops.reduce(
                    acts_fourier[:, 1:, 1:].pow(2), "freq x y neuron -> neuron", "sum"
                )
            else:
                squares_for_this_freq = einops.reduce(acts_fourier[freq - 1].pow(2), "x y neuron -> neuron", "sum")
                squares_for_this_freq_quadratic_terms = einops.reduce(
                    acts_fourier[freq - 1, 1:, 1:].pow(2), "x y neuron -> neuron", "sum"
                )

            frac_explained[neuron_freqs == freq] = squares_for_this_freq / square_of_all_terms[neuron_freqs == freq]
            frac_explained_quadratic_terms[neuron_freqs == freq] = (
                squares_for_this_freq_quadratic_terms / square_of_all_terms[neuron_freqs == freq]
            )

        returns.extend([frac_explained, frac_explained_quadratic_terms])

    frac_active = (neuron_acts > 0).float().mean(0)

    return t.nan_to_num(t.stack(returns + [neuron_freqs, frac_active], axis=0))


get_metrics(model, metric_cache, get_frac_explained, "get_frac_explained")

frac_explained_pre = metric_cache["get_frac_explained"][:, 0]
frac_explained_quadratic_pre = metric_cache["get_frac_explained"][:, 1]
frac_explained_post = metric_cache["get_frac_explained"][:, 2]
frac_explained_quadratic_post = metric_cache["get_frac_explained"][:, 3]
neuron_freqs_ = metric_cache["get_frac_explained"][:, 4]
frac_active = metric_cache["get_frac_explained"][:, 5]

utils.animate_scatter(
    t.stack([frac_explained_quadratic_pre, frac_explained_quadratic_post], dim=1)[:200:5],
    color=neuron_freqs_[:200:5],
    color_name="freq",
    snapshot="epoch",
    snapshot_index=epochs[:200:5],
    xaxis="Quad ratio pre",
    yaxis="Quad ratio post",
    title="Fraction of variance explained by quadratic terms (up to epoch 20K)",
)

utils.animate_scatter(
    t.stack([neuron_freqs_, frac_explained_pre, frac_explained_post], dim=1)[:200:5],
    color=frac_active[:200:5],
    color_name="frac_active",
    snapshot="epoch",
    snapshot_index=epochs[:200:5],
    xaxis="Freq",
    yaxis="Frac explained",
    hover=list(range(utils.d_mlp)),
    title="Fraction of variance explained by this frequency (up to epoch 20K)",
)

## 교환법칙의 발달

각 위치에 대한 평균 attention을 시각화하면, 모델이 마지막 위치에는 attention을 주지 않는 법을 빠르게 배우지만, grokking 시점이 되기 전까지는 교환법칙(즉, pos 0과 pos 1에 동일한 attention을 주는 것)을 제대로 배우지 못한다는 것을 알 수 있습니다.

**참고:** epoch 43K 부근에서 다시 position 2에 attention을 주기 시작하는 이상한 위상 변화가 나타납니다. 이 현상에 대해서는 아직 조사하지 않았습니다.

(각 프레임은 100 epoch입니다)

In [ ]:
def avg_attn_pattern(model: HookedTransformer):
    _, cache = model.run_with_cache(all_data, return_type=None)
    return to_numpy(einops.reduce(cache["pattern", 0][:, :, 2], "batch head pos -> head pos", "mean"))


get_metrics(model, metric_cache, avg_attn_pattern, "avg_attn_pattern")

utils.imshow_div(
    metric_cache["avg_attn_pattern"][::5],
    animation_frame=0,
    animation_name="head",
    title="Avg attn by position and head, snapped every 100 epochs",
    xaxis="Pos",
    yaxis="Head",
    zmax=0.5,
    zmin=0.0,
    color_continuous_scale="Blues",
    text_auto=".3f",
)

pos 0과 pos 1 사이의 평균 차이를 그래프로 그려서 이를 확인할 수도 있습니다.

In [ ]:
utils.lines(
    (metric_cache["avg_attn_pattern"][:, :, 0] - metric_cache["avg_attn_pattern"][:, :, 1]).T,
    labels=[f"Head {i}" for i in range(4)],
    x=epochs,
    xaxis="Epoch",
    yaxis="Average difference",
    title="Attention to pos 0 - pos 1 by head over training",
    width=900,
    height=450,
)

## 노이즈 제거를 위한 짧은 지연

학습 과정에 따른 test loss와 train loss를 그래프로 나타냅니다.

우리는 key frequencies의 $\cos(w(x+y)),\sin(w(x+y))$에 해당하는 logit의 방향만을 추출하여 계산한 loss를 **trig loss**로 정의합니다. 이를 전체 데이터와 학습 데이터 세트에 대해서만 각각 실행합니다.

**관찰 결과:**
* 전체 데이터에 대한 trig loss와 학습 데이터에 대한 train loss가 동일하게 나타납니다. 이는 해당 차원들이 암기(memorisation)가 아니라, train과 test를 동일하게 처리하는 일반적인 알고리즘을 위해서만 사용된다는 것을 보여줍니다.
* trig loss는 test loss가 급격히 떨어지기 전에 먼저 떨어지며, grokking 기간 동안 trig loss의 비율은 훨씬 더 낮습니다(10^4-10^5배 차이). 하지만 grokking 이후에는 다시 낮은 비율로 돌아옵니다. 이는 모델이 일반적인 알고리즘을 완전히 학습하는 단계와, memorisation circuit에 의해 남겨진 노이즈를 제거하는 단계 사이에 짧은 지연(lag)이 있음을 시사합니다.

**참고:** trig 차원으로 투영(projecting)하려면 모든 데이터를 모델에 입력해야 합니다. trig train loss를 계산하기 위해, 먼저 *모든* 데이터에 대한 logit을 구한 다음, trig 성분으로 투영하고, 그 후 test 데이터의 logit을 버립니다.

In [ ]:
def trig_loss(model: HookedTransformer, mode: str = "all"):
    logits = model(all_data)[:, -1, :-1]

    trig_logits = []
    for freq in key_freqs:
        cos_xplusy_dir, sin_xplusy_dir = get_trig_sum_directions(freq)
        cos_xplusy_proj = project_onto_direction(logits, cos_xplusy_dir.flatten())
        sin_xplusy_proj = project_onto_direction(logits, sin_xplusy_dir.flatten())
        trig_logits.extend([cos_xplusy_proj, sin_xplusy_proj])
    trig_logits = sum(trig_logits)

    return utils.test_logits(trig_logits, bias_correction=True, original_logits=logits, mode=mode)


get_metrics(model, metric_cache, trig_loss, "trig_loss")
get_metrics(model, metric_cache, partial(trig_loss, mode="train"), "trig_loss_train")

line_labels = ["test_loss", "train_loss", "trig_loss", "trig_loss_train"]
plot_metric(
    [metric_cache[lab] for lab in line_labels],
    labels=line_labels,
    title="Different losses over training",
)
plot_metric(
    [metric_cache["test_loss"] / metric_cache["trig_loss"]],
    title="Ratio of trig and test loss",
)

## 가중치 제곱합의 전개

또 다른 데이터 포인트는 각 파라미터의 가중치 제곱합을 살펴보는 것입니다. 여기에서 몇 가지 단계를 확인할 수 있습니다:

* (0-1K) 모델은 먼저 뉴런을 사용하여 암기합니다 (이 과정에서 $W_{in}$과 $W_{out}$의 전체 가중치는 크게 증가하지만, 나머지는 그렇지 않습니다).
* (1K - 8K) 그 후 모델 전체에 걸쳐 계산을 평활화하며, 이에 따라 모든 가중치 행렬이 동일한 전체 합을 갖게 됩니다. 동시에 모든 행렬의 전체 합이 감소하는데, 이는 아마도 trig 방향을 사용하는 법을 배우기 때문인 것으로 보입니다.
* (8K-13K) 이후 솔루션을 grok하게 되며 수치들이 빠르게 감소합니다. 아마도 trig 방향을 충분히 잘 사용하는 법을 배웠기 때문에, 암기에 사용되었던 다른 모든 방향들을 정리할 수 있게 된 것으로 보입니다.
* (13K-43K) 그 다음 모든 가중치가 정체기(plateau)에 진입합니다.
* (43K-) 전체 가중치 그래프를 확대해 보면 이 지점에서 작지만 눈에 띄는 굴곡이 보이며, 이는 마지막 단계의 변화입니다. (955에서 942로)

In [ ]:
def sum_sq_weights(model):
    return [param.pow(2).sum().item() for name, param in model.named_parameters()]


get_metrics(model, metric_cache, sum_sq_weights, "sum_sq_weights")

plot_metric(
    metric_cache["sum_sq_weights"].T,
    title="Sum of squared weights for each parameter",
    labels=[name.split(".")[-1] for name, _ in model.named_parameters()],
    log_y=False,
)
plot_metric(
    [einops.reduce(metric_cache["sum_sq_weights"], "epoch param -> epoch", "sum")],
    title="Total sum of squared weights",
    log_y=False,
)

# ☆ 보너스

## 토론 및 향후 방향

제가 주장하고 싶은 핵심 내용은 grokking이 일반적인 알고리즘을 학습하는 과정에서 phase change가 발생하고, *동시에* 해당 phase change가 여전히 일어날 수 있는 최소한의 데이터가 모델에 제공되었을 때 발생한다는 점입니다.

### 왜 Phase Change가 발생하는가?

저는 모델이 서로 다른 부분들이 비선형 방식으로 함께 작동해야 하는 *어떠한* 단일 일반화 circuit를 학습할 때 phase change가 발생한다고 추측합니다. 정교한 일반화 circuit를 학습하는 것은 어려우며, 모델의 서로 다른 부분들이 정확하게 정렬되어야 합니다(즉, 공유된 representation에 대해 조정되어야 합니다). 성능이 부드럽게 증가하기보다, circuit가 어떤 의미에서 작동하거나 작동하지 않는 상태 중 하나라면 이는 그리 놀라운 일이 아닙니다.

자연스럽게 이어지는 질문은 왜 모델이 일반적인 알고리즘을 학습하는가 하는 점입니다. phase change 이전의 성능이 일정하다면 gradient가 없어야 하기 때문입니다. 여기서 핵심은 *보지 못한 데이터(unseen data)*로 일반화하는 것이 어렵다는 점이라고 생각합니다. [excluded loss](#scrollTo=Excluded_Loss)에서 볼 수 있듯이, phase change 이전의 circuit들도 이미 본 데이터를 예측하는 데는 상당히 도움이 됩니다. (물론 test loss는 phase change 이전에도 감소하는 경향이 있지만, 단지 속도가 느릴 뿐입니다. 문제는 gradient가 0인지 아닌지가 아니라, 얼마나 커야 하는가에 대한 것입니다.)

저의 현재 모델은 모델의 모든 gradient update가 부분적으로는 배치 내의 데이터를 암기하려는 유인과, 부분적으로는 데이터 내의 패턴을 파악하려는 유인으로 이루어져 있다는 것입니다. 암기를 위한 gradient update는 임의의 방향을 가리키며, 모델이 완전히 암기할 만큼의 capacity가 충분하지 않다면 (배치 내부에서나 배치 간에) 서로 상쇄되는 경향이 있습니다. 반면 패턴을 식별하기 위한 gradient update는 서로를 강화합니다. Regularisation은 패턴 인식 gradient에 비해 암기 gradient를 억제하는 역할을 합니다. 무한한 데이터 설정에서는 과거의 데이터 포인트를 다시 실행하지 않기 때문에 이러한 역학을 관찰할 수 없지만, 이전 학습 데이터를 어느 정도 암기했을 것이라고 예상합니다. (예: [Does GPT-2 Know Your Phone Number?](https://bair.berkeley.edu/blog/2020/12/20/lmmem/) 참조)

저는 모델이 데이터를 가능한 한 효율적으로 암기하도록 유도되며, 즉 단순함(simplicity)을 지향하는 편향을 가지고 있기 때문이라고 추측합니다. 이는 weight decay와 같은 명시적 regularisation과, 내재적인 모델 capacity 및 SGD와 같은 암시적 regularisation 모두에서 기인합니다. 모델은 데이터 포인트 사이의 패턴을 파악하는 법을 배우는데, 이러한 규칙성들이 더 효율적인 암기를 가능하게 하기 때문입니다. 하지만 이를 위해서는 암기 circuit가 일반화 circuit(들)보다 더 복잡해질 수 있을 만큼 충분한 학습 데이터가 필요합니다.

또한, 모델이 학습할 수 있는 몇 가지 명확한(crisp) circuit가 존재해야 합니다. 만약 모델이 여러 다양한 circuit를 모호하게 학습한다면, 데이터 양을 줄임에 따라 학습하는 circuit의 수가 점점 줄어들면서 test 성능이 상당히 부드럽게 감소할 것입니다.


### 한계점

본 연구와, 이를 통해 딥러닝/grokking에 대한 일반적인 이해로 얼마나 자신 있게 일반화할 수 있는지에는 몇 가지 한계가 있습니다. 특히 다음과 같습니다:
* modular addition transformer는 toy model이며, 문제를 완전히 해결하는 데 단 몇 개의 circuit만 필요합니다 (많은 circuit가 필요한 LLM이나 이미지 모델과는 다릅니다).
* 모델이 과잉 매개변수화(over-parametrised)되어 있습니다. 이는 대략 동일한 작업을 수행하는 많은 중복 neuron을 학습하는 것에서 드러납니다. 실제 모델과 달리, 이 모델은 작업에서 거의 완벽한 성능을 내는 데 필요한 것보다 훨씬 더 많은 parameter를 가지고 있습니다.
* regularisation의 형태로 weight decay만을 연구했습니다.
* 현재 무슨 일이 일어나고 있는지에 대한 저의 설명은 명확하게 정의하지 않은 상당히 모호하고 직관적인 개념들(crisp circuits, simplicity, phase changes, 암기 vs 일반화의 의미, model capacity 등)에 크게 의존하고 있습니다.


### Alignment와의 관련성

#### 모델 학습 역학 (Model Training Dynamics)

alignment 관점에서 관련이 있어 보이는 주요 사항은 모델 학습 역학에 대한 더 나은 이해입니다.

특히, phase change는 alignment 관점에서 상당히 부정적으로 보입니다. 왜냐하면 모델이 학습되거나 scale-up 될 때 급격한 능력 향상이 일어날 가능성이 높음을 시사하기 때문입니다. 이는 위험한 unaligned 시스템이 나타나기 전에 경고 신호(즉, 정교하지 않은 unaligned 행동을 보이는 시스템)를 받을 가능성을 낮추며, AGI에 근접한 시스템이 alignment 연구를 위한 좋은 경험적 테스트 베드가 될 가능성을 낮춥니다. 본 연구는 phase change의 더 많은 사례를 보여주며, alignment가 어려울 것이라는 저의 생각을 약간 더 강화합니다.

동시에 이 연구는 저에게 약간의 희망을 줍니다. phase change가 일어나기 전에 모델이 미리 circuit를 학습함으로써 분명하게 예고된다는 점과, interpretability에서 영감을 받은 metric으로 능력 향상을 예측할 수 있다는 점을 보여주기 때문입니다 (물론 이는 toy case에서만 나타난 것이며, 제가 circuit를 충분히 이해한 후에야 가능했습니다. misalignment를 식별하기 위해 unaligned AGI를 직접 학습시키고 해석해야 하는 상황은 피하고 싶습니다!).

더 추측하자면, 이는 우리가 interpretability 도구를 사용하여 학습 역학을 형성할 수 있음을 시사합니다. 여기서 자연스러운 우려는 모델이 우리의 도구를 기만(obfuscate)하는 법을 배워서, 도구에는 나타나지 않으면서 동일한 알고리즘을 수행할 수 있다는 점입니다. 이는 gradient descent가 직접적으로 기만을 학습하거나(unaligned 솔루션이 aligned 솔루션보다 훨씬 성능이 좋기 때문에), 시스템 자체가 [learned to be deceptive and alter itself to avoid our tools](https://www.lesswrong.com/posts/nbq2bWLcYmSGup9aF/a-transparency-and-interpretability-tech-tree)를 가짐으로써 발생할 수 있습니다. circuit가 일반화 능력을 갖추기 훨씬 전부터 발달하는 것을 관찰했다는 사실은, 모델이 일반화하여 기만 탐지기를 회피할 수 있을 만큼 정교해지기 전에 기만하려는 유인을 제거할 수 있을지도 모른다는 점을 시사합니다 (다만 gradient descent를 회피하는 것에는 큰 도움이 되지 않습니다).

자연스러운 향후 방향은 interpretability 기반 metric으로 학습하는 것을 탐구하고, gradient descent가 이를 Goodhart-ing(수치 최적화에만 치중) 하는지, 아니면 다른 알고리즘을 배우도록 inductive bias를 전환하는지 확인하는 것입니다. 예를 들어, 일반화를 유도하여 더 적은 데이터로 grokking을 이끌어낼 수 있을까요? 암기를 유도하여 학습하는 알고리즘을 바꿀 수 있을까요? 특정 주파수로 덧셈을 배우는 것을 억제하면 어떤 일이 벌어질까요?

#### 기타 관련성

또한 이는 네트워크가 기계적으로 이해될 수 있으며 해석 가능한 알고리즘을 학습하고 있다는 [circuits hypothesis](https://distill.pub/2020/circuits/zoom-in/)의 추가적인 증거로서 관련이 있어 보입니다. 그리고 transformer의 neuron을 해석한 초기 사례 중 하나라는 점에서도 의미가 있습니다 (비록 언어와는 완전히 다른 작업이지만 말입니다).

또한 모델이 어떻게 일반화하는지, 그리고 grokking이라는 기이한 현상을 더 잘 이해함으로써 딥러닝 과학을 발전시킨다는 점에서도 관련이 있습니다. 모델이 aligned 될지 여부를 알기 위해서는 모델이 어떻게 일반화할지, 미래의 모델 행동은 어떠할지, 그리고 어떤 학습 설정이 일반화되고 되지 않을지를 이해해야 합니다. 이러한 동기는 다소 분산되어 있고 덜 직접적이지만, 통계적 학습 이론이 고전 통계학의 모델에 대해 예측할 수 있게 해주는 것과 유사해 보입니다 (물론 딥러닝에는 분명히 불충분합니다).

(솔직히 말씀드리면, 저는 주로 이 작업이 재미있었고, 휴가 중이었으며, 이 문제에 완전히 매료되었기 때문에 연구했습니다. 따라서 이 모든 내용은 순수하게 alignment를 목적으로 한 연구라기보다, 다소 임시방편적이고 사후 분석적인 성격이 강합니다.)


### 학습 역학 (Training Dynamics)

이 연구가 통찰을 주는 흥미로운 점 중 하나는 circuit의 학습 역학입니다. 즉, circuit의 어느 부분이 먼저 발달하고, 어떤 속도로, 왜 발달하는가 하는 점입니다.

저의 추측 섞인 가설은, circuit가 레이어 간에 상호작용하는 여러 구성 요소를 가지고 있을 때, logit에 가장 가까운 구성 요소가 형성되기 더 쉬우며 먼저 형성되는 경향이 있다는 것입니다.

**직관:** 중간에 비선형성(non-linearity)을 두고 상호작용하는 두 구성 요소로 이루어진 circuit가 있고, 둘 다 무작위로 초기화되었다고 가정해 보겠습니다. 이들은 hidden space의 몇 가지 방향을 포함하는 공유된 representation으로 수렴하고자 합니다. 처음에는 무작위인 첫 번째 구성 요소가 올바른 feature에 해당하는 출력을 일부 내놓겠지만, 계수가 작고 많은 노이즈가 섞여 있을 것입니다. 두 번째 구성 요소는 이러한 올바른 feature에 집중하고 노이즈를 제거하는 법을 배울 수 있으며, 이는 첫 번째 구성 요소가 해당 feature에 집중하도록 하는 유인을 강화합니다. 반면, 두 번째 구성 요소가 무작위라면, 첫 번째 구성 요소가 두 번째 구성 요소에 의해 생산적으로 사용될 수 있는 합리적인 feature를 만들어내기가 어렵습니다.

정성적으로 관찰했을 때, 모든 circuit는 grokking 이전에 병렬적으로 형성되지만, 그 순서는 대략 logit circuit > embedding circuit > neuron circuit > attention circuit 순으로 보입니다 (즉, logit이 가장 빠르고 attention이 가장 느립니다).

이는 학습 중에 연구하기 쉬운 명확한(crisp) circuit의 구체적인 사례를 제공함으로써 향후 연구의 흥미로운 방향이 될 것으로 보입니다. 가능한 초기 실험으로는 네트워크의 일부를 무작위로 고정하거나, 일부를 완전히 학습된 상태로 초기화하여 동결시키거나, 네트워크의 각 부분에 서로 다른 learning rate를 부여하는 것 등이 있을 것입니다.

## 추천 캡스톤 프로젝트

### phase change 조사하기

다음과 같은 phase change의 다른 사례들을 찾아볼 수 있습니다:

* Toy 문제
    * skip trigram을 유도하는 설정
    * virtual attention head를 유도하는 설정
    * 예: 아래 모델 중 하나 (또는 더 쉬운 태스크 선택)
* ConvNet에서 [curve detectors](https://distill.pub/2020/circuits/curve-circuits) 찾기
    * 이를 시도하는 단순한 방법은 Inception의 실제 curve detector를 모방하도록 모델을 학습시키는 것입니다 (예: 모델의 출력과 curve detector activation 사이의 OLS loss 최소화)
* [SoLU transformer](https://transformer-circuits.pub/2022/solu/index.html)에서 interpretable neuron의 형성 관찰하기
* 많은 checkpoint가 있는 LLM 내부 살펴보기
    * Eleuther는 GPT-J와 GPT-Neo의 많은 checkpoint를 보유하고 있으며, 요청 시 공유해 줄 것입니다
    * [Mistral](https://nlp.stanford.edu/mistral/getting_started/download.html)는 5번의 run과 많은 checkpoint가 포함된 GPT-2 small 및 medium의 공개 버전을 제공합니다
    * 조사 가능한 capability 예시
        * benchmark 성능 또는 benchmark의 특정 질문에 대한 답변
        * 덧셈, 단어 알파벳 순 정렬, 괄호 짝 맞추기와 같은 간단한 알고리즘 태스크
        * Soft induction head, 예: [translation](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html#performing-translation)
        * 다양한 텍스트에 대한 attention head를 살펴보고, 인식 가능한 attention pattern이 있는지 확인합니다 (예: 단어의 시작, 현재 단어를 수식하는 형용사, 들여쓰기나 변수 정의와 같은 코드의 구문적 특징, 가장 최근의 열린 괄호 등).

### 더 많은 알고리즘 문제 시도하기

toy model을 해석하는 것은 TransformerLens와 기본적인 interpretability 방법론을 사용하는 자신감을 높이는 좋은 방법입니다. mechanistic interpretability의 공개 문제들 중 가장 흥미로운 카테고리는 아닐 수 있지만, 여전히 유용한 연습이 될 수 있으며, 때로는 interpretability 도구를 어떻게 사용할 수 있는지에 대한 흥미로운 새로운 통찰로 이어질 수 있습니다.

원하신다면 LeetCode에 접속하여 적절한 문제("Easy" 섹션을 추천합니다)를 선택해 transformer를 학습시키고 그 출력을 해석해 볼 수 있습니다. 시작을 돕기 위한 몇 가지 제안은 다음과 같습니다 (일부는 LeetCode에서, 다른 일부는 Neel Nanda의 [open problems post](https://www.lesswrong.com/s/yivyHaCAmMJ3CqSyj/p/ejtFsvyhRkMofKAFy)에서 가져왔습니다). 제가 직접 해석해 본 것은 아니기에 추측일 뿐이지만, 대략 쉬운 순서에서 어려운 순서로 나열했습니다. 참고로, 수정을 통해 이러한 문제들을 더 쉽게 또는 더 어렵게 만들 수 있는 방법들이 있으며, 몇 가지 아이디어를 본문에 포함했습니다.

* 피보나치 스타일의 재귀 관계를 가진 시퀀스 계산 (즉, 이전 두 요소로부터 다음 요소를 예측)
* [Search Insert Position](https://leetcode.com/problems/search-insert-position/) - 타겟이 항상 리스트에 포함되는 것이 보장되는 경우가 더 쉬운 버전입니다 (이 경우 정렬에 대해 걱정할 필요가 없습니다). 이러한 보장이 없는 버전은 매우 다른 문제이며 훨씬 더 어려울 것입니다.
* [Is Subsequence](https://leetcode.com/problems/is-subsequence/) - 길이 1의 subsequence부터 시작하여 (이 경우 이전 문제의 쉬운 버전과 매우 유사합니다), 점차 난이도를 높여가야 합니다.
* [Majority Element](https://leetcode.com/problems/majority-element/) - 데이터 생성 과정을 조정하여 난이도를 변경해 볼 수 있습니다. 예를 들어, 다수 요소의 빈도에 대한 보장이 없는 시퀀스 (즉, 단순히 다른 어떤 token보다 더 많이 나타나는 token을 찾는 경우)는 훨씬 더 어려울 것입니다.
* [Number of Equivalent Domino Pairs](https://leetcode.com/problems/number-of-equivalent-domino-pairs/) - 문제를 매우 짧은 domino 리스트로 제한하여 더 쉽게 만들 수 있습니다 (예: 단 2개의 domino로 시작!).
* [Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/)
* [Isomorphic Strings](https://leetcode.com/problems/isomorphic-strings/) - 첫 번째 문자열에만 중복 문자를 허용하거나, 문자열 길이/vocabulary 크기를 제한하여 더 단순하게 만들 수 있습니다.
* [Plus One](https://leetcode.com/problems/plus-one/) - 이를 시도하기 전에 "sum of numbers" 알고리즘 문제나 이 장의 grokking 연습 문제를 살펴보는 것이 좋습니다. 이 문제를 잘 이해하면 실제로 "sum of numbers" 문제를 해석하는 단계로 나아가는 데 도움이 될 수 있습니다 (저는 이를 수행하지 않았으므로, 제가 carry 메커니즘을 깊게 파고들지 않았기 때문에 [mine](https://www.perfectlynormal.co.uk/blog-november-monthly-problem)보다 해당 월간 문제에 대해 더 나은 해석을 찾아내실 가능성이 매우 높습니다).
* permutation 예측, 즉 12-token 시퀀스 `(17 3 11) (17 1 13) (11 2 4) (11 4 2)`의 마지막 3개 token을 예측하는 것입니다 (즉, 모델은 첫 번째 그룹에서 두 번째 그룹을 얻기 위해 어떤 permutation 함수가 적용되었는지 학습하고, 그 permutation을 세 번째 그룹에 적용하여 네 번째 그룹을 정확하게 예측해야 합니다). 참고로, 이 문제를 해결하려면 3개의 layer가 필요할 수 있습니다. 그 이유를 알 수 있을까요?
* [automata](https://arxiv.org/pdf/2210.10749.pdf) task를 위한 모델을 학습시키고 해석해 보십시오. 결과가 이론과 일치합니까?
* 간단한 코드 함수의 출력 예측. 예를 들어, 다음 시퀀스에서 `1 2 4` 텍스트를 예측하는 것입니다 (이는 당연히 몇 가지 명백한 수정, 예를 들어 변수 정의를 더 추가하여 모델이 올바른 변수에 attention을 기울여야 하게 함으로써 더 어렵게 만들 수 있습니다):
```python
a = 1 2 3
a[2] = 4
a -> 1 2 4
```

* [this](https://jacobbrazeal.wordpress.com/2022/09/23/gpt-3-can-find-paths-up-to-7-nodes-long-in-random-graphs/)과 같은 그래프 이론 문제. 이러한 task로 transformer를 학습시킬 때는 입력 형식에 대해 창의력을 발휘해야 할 수도 있습니다!

참고로, ARENA는 [monthly algorithmic problems sequence](https://arena-ch1-transformers.streamlit.app/Monthly_Algorithmic_Problems)를 운영하고 있으며, 이 시리즈의 지난 문제들을 살펴보며 아이디어를 얻을 수 있습니다. 또한 이러한 repo들을 사용하여 toy model에서 transformer를 구축 및 학습시키고, 특정 문제에 맞는 데이터셋을 구성하기 위한 샘플 코드를 얻을 수 있습니다.

## 추천 논문 구현 과제

### [A Toy Model of Universality: Reverse Engineering How Networks Learn Group Operations](https://arxiv.org/abs/2302.03025)

이 논문은 modular addition의 특수한 사례인 일반적인 group operation을 살펴봄으로써, 이 특정 task와 model에 대한 분석을 확장합니다.

다음과 같은 경우에 이 논문을 복제(replication)해 보는 것이 좋습니다:

* 이 연습 세트의 모든 하위 섹션을 즐겁게 학습했으며, 더 복잡한 알고리즘에 대해 유사한 분석을 수행하고 싶은 경우
* grokking 및 training dynamics 연구에 관심이 있는 경우
* 수학적 배경지식이 있으며, 특히 group theory(그리고 가급적 representation theory)에 익숙한 경우